In [5]:
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", None)

# ======================================================
# CAMINHOS DO PROJETO
# ======================================================

# O notebook está em:
# article_5/src/notebooks/experiments.ipynb

PROJECT_ROOT = Path.cwd().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = SRC_DIR / "data"
OUTPUTS_DIR = SRC_DIR / "outputs"

# Garantir que article_5 está disponível para imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR existe:", DATA_DIR.exists())

print("\nFicheiros disponíveis:")
for file in sorted(DATA_DIR.iterdir()):
    print("-", file.name)

PROJECT_ROOT: c:\Users\msnev\article_5
DATA_DIR: c:\Users\msnev\article_5\src\data
DATA_DIR existe: True

Ficheiros disponíveis:
- dataframe_hotel_id_week_year_meteo_google_trend.csv
- dataset_sentiment_and_emotion_analysis_year_week.csv
- dataset_sentiment_and_emotion_analysis_year_week_artigo4.csv
- dataset_sentiment_and_emotion_analysis_year_week_artigo4_info_expectation.csv
- dataset_sentiment_and_emotion_analysis_year_week_artigo4_info_expectation_gpt5.csv
- df_info_hotel_ocupacao_kpis_weekly.csv
- hotels_info_extra.csv
- id_hotels_and_info_extra_and_reviews.csv
- reviews_by_hotel_translated.parquet
- translation_checkpoints


In [6]:
# ======================================================
# LEITURA DOS DATASETS
# ======================================================

df_info_hotel_ocupacao_kpis_weekly = pd.read_csv(
    DATA_DIR / "df_info_hotel_ocupacao_kpis_weekly.csv",
    low_memory=False
)

dataset_sentiment_and_emotion_analysis_year_week = pd.read_csv(
    DATA_DIR / "dataset_sentiment_and_emotion_analysis_year_week_artigo4_info_expectation_gpt5.csv",
    low_memory=False
)

dataset_sentiment_and_emotion_analysis_year_week_artigo3 = pd.read_csv(
    DATA_DIR / "dataset_sentiment_and_emotion_analysis_year_week.csv",
    low_memory=False
)

#reviews_by_hotel = pd.read_csv(
#    DATA_DIR / "id_hotels_and_info_extra_and_reviews.csv",
#    low_memory=False
#)
reviews_by_hotel = pd.read_parquet(
    DATA_DIR / "reviews_by_hotel_translated.parquet"
)

info_extra_hotel = pd.read_csv(
    DATA_DIR / "hotels_info_extra.csv",
    sep=";",
    usecols=["nome_hotel_url", "ID_HOTEL"],
    low_memory=False
)

print("KPIs:", df_info_hotel_ocupacao_kpis_weekly.shape)
print("Experiential variables:", dataset_sentiment_and_emotion_analysis_year_week.shape)
print("Sentiment/emotion:", dataset_sentiment_and_emotion_analysis_year_week_artigo3.shape)
print("Reviews:", reviews_by_hotel.shape)
print("Informação dos hotéis:", info_extra_hotel.shape)

KPIs: (58778, 21)
Experiential variables: (41097, 73)
Sentiment/emotion: (41097, 38)
Reviews: (452061, 34)
Informação dos hotéis: (548, 2)


In [7]:
import os
import json
import time
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from dotenv import load_dotenv

from openai import OpenAI

AZURE_API_KEY = "BYh6Xij0td0cPnTcdbF5Gb4fPCKaqKM0IDxe9ZeVlN49Kk5gZqPjJQQJ99CBACYeBjFXJ3w3AAABACOG7Z9d"

client = OpenAI(
    api_key=AZURE_API_KEY,
    base_url="https://ai-pgbiht.openai.azure.com/openai/v1"
)

MODEL = "gpt-5-mini"


c:\Users\msnev\anaconda3\envs\llama_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tradução

In [4]:
response = client.responses.create(
    model=MODEL,
    input="Translate to English: Porta do banheiro necessita de manutenção."
)

print(response.output_text)

"The bathroom door needs maintenance." 

(Alternative: "The bathroom door needs repair.")


In [5]:
import os
import json
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from dotenv import load_dotenv


# ============================================================
# CONFIGURAÇÃO
# ============================================================

CHECKPOINT_DIR = Path("../data/translation_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


FINAL_OUTPUT_PATH = Path(
    "../data/reviews_by_hotel_translated.parquet"
)


columns_to_translate = [
    "review_text_liked",
    "review_text_disliked",
]


# Começaria com 30.
# Se continuar muito lento, podemos posteriormente otimizar.
batch_size = 15

max_retries = 3


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def clean_text(value):
    """Transforma valores válidos em string limpa."""

    if pd.isna(value):
        return None

    value = str(value).strip()

    if not value:
        return None

    if value.lower() in {"nan", "none"}:
        return None

    return value


def translate_batch_openai(
    texts,
    max_retries=3
):
    """
    Traduz um batch de textos para inglês usando
    Structured Outputs.
    """

    items = [
        {
            "id": i,
            "text": text
        }
        for i, text in enumerate(texts)
    ]

    prompt = f"""
Translate all hotel review texts below into English.

Rules:
- Preserve the original meaning.
- Do not summarize.
- Do not add explanations.
- If the text is already English, return it unchanged.
- Preserve names and proper nouns.
- Return exactly one translation per input item.
- Preserve each id exactly.

Input:
{json.dumps(items, ensure_ascii=False)}
"""

    schema = {
        "type": "object",
        "properties": {
            "translations": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "id": {
                            "type": "integer"
                        },
                        "translation": {
                            "type": "string"
                        }
                    },
                    "required": [
                        "id",
                        "translation"
                    ],
                    "additionalProperties": False
                }
            }
        },
        "required": [
            "translations"
        ],
        "additionalProperties": False
    }

    last_error = None

    for attempt in range(
        1,
        max_retries + 1
    ):

        try:

            response = client.responses.create(
                model=MODEL,
                input=prompt,

                text={
                    "format": {
                        "type": "json_schema",
                        "name": "translation_batch",
                        "schema": schema,
                        "strict": True
                    }
                }
            )

            raw = response.output_text

            if not raw:
                raise ValueError(
                    "Empty response from API."
                )

            result = json.loads(raw)

            result_items = result[
                "translations"
            ]

            translations = {
                int(item["id"]):
                item["translation"]
                for item in result_items
            }

            expected_ids = set(
                range(len(texts))
            )

            received_ids = set(
                translations.keys()
            )

            if received_ids != expected_ids:

                missing = (
                    expected_ids
                    - received_ids
                )

                extra = (
                    received_ids
                    - expected_ids
                )

                raise ValueError(
                    f"Invalid IDs. "
                    f"Missing={sorted(missing)}, "
                    f"Extra={sorted(extra)}"
                )

            return [
                translations[i]
                for i in range(
                    len(texts)
                )
            ]

        except Exception as error:

            last_error = error

            print(
                f"\nTentativa "
                f"{attempt}/{max_retries} "
                f"falhou: {error}"
            )

            if attempt < max_retries:

                wait_time = 5 * attempt

                print(
                    f"Aguardar "
                    f"{wait_time}s..."
                )

                time.sleep(
                    wait_time
                )

    raise RuntimeError(
        "Batch falhou após "
        f"{max_retries} tentativas. "
        f"Último erro: {last_error}"
    )


def translate_individually(text):
    """
    Fallback para quando um batch falha repetidamente.
    """

    try:

        result = translate_batch_openai(
            [text],
            max_retries=2
        )

        return result[0]

    except Exception as error:

        print(
            f"\nFalha individual: {error}"
        )

        return None


def save_checkpoint(
    translation_dict,
    checkpoint_path
):
    """Guarda traduções já realizadas."""

    checkpoint_df = pd.DataFrame(
        {
            "original": list(
                translation_dict.keys()
            ),
            "english": list(
                translation_dict.values()
            )
        }
    )

    checkpoint_df.to_parquet(
        checkpoint_path,
        index=False
    )

In [6]:
# ============================================================
# PREPARAÇÃO DA LÍNGUA
# ============================================================

reviews_by_hotel["original_lang"] = (
    reviews_by_hotel["original_lang"]
    .astype("string")
    .str.lower()
    .str.strip()
)


for column in columns_to_translate:

    print("\n")
    print("=" * 70)
    print(f"COLUNA: {column}")
    print("=" * 70)

    output_column = f"en_{column}"

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"{column}_translations.parquet"
    )

    # ========================================================
    # 1. TEXTOS VÁLIDOS
    # ========================================================

    original_clean = (
        reviews_by_hotel[column]
        .apply(clean_text)
    )

    # ========================================================
    # 2. TEXTOS EM INGLÊS
    # ========================================================

    english_mask = (
        reviews_by_hotel["original_lang"]
        .eq("en")
        & original_clean.notna()
    )

    print(
        f"Reviews em inglês: "
        f"{english_mask.sum():,}"
    )

    # ========================================================
    # 3. CARREGAR CHECKPOINT
    # ========================================================

    if checkpoint_path.exists():

        checkpoint = pd.read_parquet(
            checkpoint_path
        )

        # remover eventuais valores vazios
        checkpoint = checkpoint[
            checkpoint["original"].notna()
            & checkpoint["english"].notna()
        ].copy()

        translation_dict = dict(
            zip(
                checkpoint["original"],
                checkpoint["english"]
            )
        )

        print(
            f"Checkpoint encontrado: "
            f"{len(translation_dict):,} traduções"
        )

    else:

        translation_dict = {}

        print(
            "Nenhum checkpoint anterior encontrado."
        )

    # ========================================================
    # 4. INGLÊS NÃO PRECISA DE API
    # ========================================================

    english_texts = (
        original_clean[
            english_mask
        ]
        .drop_duplicates()
    )

    for text in english_texts:

        translation_dict[text] = text

    print(
        f"Textos ingleses adicionados diretamente: "
        f"{len(english_texts):,}"
    )

    # ========================================================
    # 5. APENAS TEXTOS NÃO INGLESES
    # ========================================================

    non_english_mask = (
        ~reviews_by_hotel["original_lang"].eq("en")
        & original_clean.notna()
    )

    unique_non_english = (
        original_clean[
            non_english_mask
        ]
        .drop_duplicates()
    )

    print(
        f"Textos únicos não ingleses: "
        f"{len(unique_non_english):,}"
    )

    # ========================================================
    # 6. RETIRAR OS QUE JÁ ESTÃO NO CHECKPOINT
    # ========================================================

    texts_to_translate = [
        text
        for text in unique_non_english.tolist()
        if text not in translation_dict
    ]

    print(
        f"Faltam traduzir via API: "
        f"{len(texts_to_translate):,}"
    )

    # ========================================================
    # 7. TRADUÇÃO
    # ========================================================

    failed_texts = []

    total_batches = (
        len(texts_to_translate)
        + batch_size - 1
    ) // batch_size

    for batch_number, i in enumerate(
        tqdm(
            range(
                0,
                len(texts_to_translate),
                batch_size
            ),
            total=total_batches,
            desc=f"Traduzindo {column}"
        ),
        start=1
    ):

        batch = texts_to_translate[
            i:i + batch_size
        ]

        try:

            translated = (
                translate_batch_openai(
                    batch,
                    max_retries=max_retries
                )
            )

            for original, english in zip(
                batch,
                translated
            ):

                if english is not None:

                    english = str(
                        english
                    ).strip()

                    if english:

                        translation_dict[
                            original
                        ] = english

        except Exception as error:

            print(
                f"\nBatch {batch_number} "
                f"falhou definitivamente:"
            )

            print(error)

            print(
                "A tentar os textos "
                "individualmente..."
            )

            # ----------------------------------------------
            # fallback individual
            # ----------------------------------------------

            for text in batch:

                english = (
                    translate_individually(
                        text
                    )
                )

                if english:

                    translation_dict[
                        text
                    ] = english

                else:

                    failed_texts.append(
                        text
                    )

        # ====================================================
        # 8. CHECKPOINT A CADA 10 BATCHES
        # ====================================================

        if batch_number % 10 == 0:

            save_checkpoint(
                translation_dict,
                checkpoint_path
            )

    # ========================================================
    # 9. CHECKPOINT FINAL
    # ========================================================

    save_checkpoint(
        translation_dict,
        checkpoint_path
    )

    # ========================================================
    # 10. MAPEAR PARA O DATAFRAME ORIGINAL
    # ========================================================

    reviews_by_hotel[
        output_column
    ] = (
        original_clean
        .map(translation_dict)
        .astype("string")
    )

    # ========================================================
    # 11. ESTATÍSTICAS
    # ========================================================

    valid_original = (
        original_clean.notna()
    )

    translated_ok = (
        reviews_by_hotel[
            output_column
        ].notna()
        & valid_original
    )

    missing_translation = (
        valid_original
        & reviews_by_hotel[
            output_column
        ].isna()
    )

    print("\nRESULTADO")
    print("-" * 50)

    print(
        f"Textos válidos: "
        f"{valid_original.sum():,}"
    )

    print(
        f"Textos com versão inglesa: "
        f"{translated_ok.sum():,}"
    )

    print(
        f"Textos ainda sem tradução: "
        f"{missing_translation.sum():,}"
    )

    print(
        f"Falhas definitivas nesta execução: "
        f"{len(failed_texts):,}"
    )


# ============================================================
# 12. GUARDAR DATASET COMPLETO
# ============================================================

reviews_by_hotel.to_parquet(
    FINAL_OUTPUT_PATH,
    index=False
)

print("\n")
print("=" * 70)
print("TRADUÇÃO TERMINADA")
print("=" * 70)

print(
    f"Dataset guardado em:\n"
    f"{FINAL_OUTPUT_PATH}"
)



COLUNA: review_text_liked
Reviews em inglês: 47,553
Checkpoint encontrado: 60,230 traduções
Textos ingleses adicionados diretamente: 44,829
Textos únicos não ingleses: 179,713
Faltam traduzir via API: 163,721


Traduzindo review_text_liked:  27%|██▋       | 2943/10915 [14:17:33<46:50:39, 21.15s/it]


Tentativa 1/3 falhou: Unterminated string starting at: line 1 column 1763 (char 1762)
Aguardar 5s...

Tentativa 2/3 falhou: Unterminated string starting at: line 1 column 2008 (char 2007)
Aguardar 10s...

Tentativa 3/3 falhou: Expecting value: line 1 column 1 (char 0)

Batch 2944 falhou definitivamente:
Batch falhou após 3 tentativas. Último erro: Expecting value: line 1 column 1 (char 0)
A tentar os textos individualmente...


Traduzindo review_text_liked:  30%|██▉       | 3258/10915 [15:55:08<36:08:16, 16.99s/it] 


Tentativa 1/3 falhou: Error code: 400 - {'error': {'message': 'The response was filtered due to the prompt triggering Azure OpenAI’s content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766', 'type': 'invalid_request_error', 'param': 'prompt', 'code': 'content_filter', 'content_filters': [{'blocked': True, 'source_type': 'prompt', 'content_filter_raw': [], 'content_filter_results': {'hate': {'filtered': True, 'severity': 'medium'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}}, 'content_filter_offsets': {'start_offset': 0, 'end_offset': 4659, 'check_offset': 0}}], 'innererror': {'code': 'ContentFiltered'}}}
Aguardar 5s...

Tentativa 2/3 falhou: Error code: 400 - {'error': {'message': 'The

...


Tentativa 1/3 falhou: Error code: 400 - {'error': {'message': 'The response was filtered due to the prompt triggering Azure OpenAI’s content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766', 'type': 'invalid_request_error', 'param': 'prompt', 'code': 'content_filter', 'content_filters': [{'blocked': True, 'source_type': 'prompt', 'content_filter_raw': [], 'content_filter_results': {'hate': {'filtered': True, 'severity': 'medium'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}}, 'content_filter_offsets': {'start_offset': 0, 'end_offset': 3870, 'check_offset': 0}}], 'innererror': {'code': 'ContentFiltered'}}}
Aguardar 5s...

Tentativa 2/3 falhou: Error code: 400 - {'error': {'message': 'The

Traduzindo review_text_disliked:  98%|█████████▊| 7430/7579 [40:24:50<49:04, 19.76s/it]  


Tentativa 1/3 falhou: Expecting value: line 1 column 1 (char 0)
Aguardar 5s...

Tentativa 2/3 falhou: Expecting value: line 1 column 1 (char 0)
Aguardar 10s...

Tentativa 3/3 falhou: Expecting value: line 1 column 1 (char 0)

Batch 7431 falhou definitivamente:
Batch falhou após 3 tentativas. Último erro: Expecting value: line 1 column 1 (char 0)
A tentar os textos individualmente...

Tentativa 1/2 falhou: Expecting value: line 1 column 1 (char 0)
Aguardar 5s...

Tentativa 2/2 falhou: Unterminated string starting at: line 5 column 22 (char 64)

Falha individual: Batch falhou após 2 tentativas. Último erro: Unterminated string starting at: line 5 column 22 (char 64)


Traduzindo review_text_disliked: 100%|██████████| 7579/7579 [41:13:39<00:00, 19.58s/it]  



RESULTADO
--------------------------------------------------
Textos válidos: 168,094
Textos com versão inglesa: 168,083
Textos ainda sem tradução: 11
Falhas definitivas nesta execução: 11


TRADUÇÃO TERMINADA
Dataset guardado em:
..\data\reviews_by_hotel_translated.parquet


In [7]:
# ============================================================
# 1. QUANTOS FICARAM SEM TRADUÇÃO?
# ============================================================

for column in [
    "review_text_liked",
    "review_text_disliked",
]:
    en_column = f"en_{column}"

    valid_original = (
        reviews_by_hotel[column]
        .notna()
        & reviews_by_hotel[column]
        .astype(str)
        .str.strip()
        .ne("")
    )

    missing_translation = (
        valid_original
        & (
            reviews_by_hotel[en_column].isna()
            | reviews_by_hotel[en_column]
            .astype("string")
            .str.strip()
            .eq("")
        )
    )

    print("=" * 60)
    print(column)
    print(f"Textos válidos: {valid_original.sum():,}")
    print(
        f"Traduzidos: "
        f"{(valid_original & ~missing_translation).sum():,}"
    )
    print(
        f"Por traduzir: "
        f"{missing_translation.sum():,}"
    )
    print(
        f"% traduzido: "
        f"{100 * (valid_original & ~missing_translation).sum() / valid_original.sum():.4f}%"
    )

review_text_liked
Textos válidos: 246,934
Traduzidos: 246,930
Por traduzir: 4
% traduzido: 99.9984%
review_text_disliked
Textos válidos: 168,148
Traduzidos: 168,083
Por traduzir: 65
% traduzido: 99.9613%


In [8]:
missing_rows = []

for column in [
    "review_text_liked",
    "review_text_disliked",
]:
    en_column = f"en_{column}"

    mask = (
        reviews_by_hotel[column].notna()
        & reviews_by_hotel[column]
        .astype(str)
        .str.strip()
        .ne("")
        & (
            reviews_by_hotel[en_column].isna()
            | reviews_by_hotel[en_column]
            .astype("string")
            .str.strip()
            .eq("")
        )
    )

    temp = reviews_by_hotel.loc[
        mask,
        [
            "original_lang",
            column,
            en_column
        ]
    ].copy()

    temp["source_column"] = column

    missing_rows.append(temp)

missing_df = pd.concat(
    missing_rows,
    ignore_index=True
)

display(missing_df)

print(
    "Total de textos ainda sem tradução:",
    len(missing_df)
)

,original_lang,review_text_liked,en_review_text_liked,source_column,review_text_disliked,en_review_text_disliked
0,en-us,none,<NA>,review_text_liked,NaN,<NA>
1,es,"lo que n me gusto ,, fue los grupos de britanicos .. ya que no respetan al resto de los clientes.. son muy incivicos .. y actuan como si estuvieran solos .. ya que la gente tiene el derecho de estar tranquilos ,,",<NA>,review_text_liked,NaN,<NA>
2,en,none,<NA>,review_text_liked,NaN,<NA>
3,pt,Barulho a noite toda. Brasileiros a falar alto as 4 da manhã como se fosse de manhã. Gaveta no quarto com anal plug e vibradores. Nojento,<NA>,review_text_liked,NaN,<NA>
4,en-us,NaN,<NA>,review_text_disliked,none,<NA>
5,pl,NaN,<NA>,review_text_disliked,Otrzymaliśmy inny pokój niż zarezerwowaliśmy. Po interwencji został zmieniony. W pokoju czajnik ale brak kawy i herbaty. Na śniadaniu szału nie ma. Porto Moniz oblegane przez imigrantów łącznie z wejściem do metra - odstraszające wrażenie. Oni tam koczują i załatwiają swoje potrzeby fizjologiczne.,<NA>
6,en,NaN,<NA>,review_text_disliked,none,<NA>
7,en-us,NaN,<NA>,review_text_disliked,none,<NA>
8,en,NaN,<NA>,review_text_disliked,none,<NA>
9,en-us,NaN,<NA>,review_text_disliked,"Location is sketchy, filled with bad immigrants. However, it was close to the city center and we didn’t have any issues.",<NA>


Total de textos ainda sem tradução: 69


In [9]:
sample_liked = (
    reviews_by_hotel[
        reviews_by_hotel["original_lang"].ne("en")
        & reviews_by_hotel["review_text_liked"].notna()
        & reviews_by_hotel["en_review_text_liked"].notna()
    ][
        [
            "original_lang",
            "review_text_liked",
            "en_review_text_liked"
        ]
    ]
    .sample(
        n=20,
        random_state=42
    )
)

display(sample_liked)

,original_lang,review_text_liked,en_review_text_liked
230482,es,Personal agradable. Restaurante y limpieza,Nice staff. Restaurant and cleanliness
414958,pt,Da localização,The location.
278155,pt,O tamanho da cama,The size of the bed.
102892,en-us,"It was very safety, and clean a place I love the definition of go back💯","It was very safety, and clean a place I love the definition of go back💯"
290071,he,"הבריכה , ארוחת בוקר","The pool, breakfast"
186538,es,Todo excelente,Everything excellent
54497,en-us,"Great location, very central and close to Metro.","Great location, very central and close to Metro."
110658,fr,Petit déjeuner un peu répétitif,Breakfast a bit repetitive
26774,fr,super petit déjeuner et personnel très gentil ! Magnifique ambiance d’époque dans l’hôtel.,Great breakfast and very kind staff! Magnificent period ambiance in the hotel.
166192,en-us,"Good, but we expected mote typical Portuguese food","Good, but we expected mote typical Portuguese food"


In [10]:
sample_disliked = (
    reviews_by_hotel[
        reviews_by_hotel["original_lang"].ne("en")
        & reviews_by_hotel["review_text_disliked"].notna()
        & reviews_by_hotel["en_review_text_disliked"].notna()
    ][
        [
            "original_lang",
            "review_text_disliked",
            "en_review_text_disliked"
        ]
    ]
    .sample(
        n=20,
        random_state=42
    )
)

display(sample_disliked)

,original_lang,review_text_disliked,en_review_text_disliked
91684,de,Kleines Zimmer,Small room.
58292,en-us,"Not the best location in and of itself. Close to public transportation, though, and within easy walking distance of the Parque Eduardo VII and Jardim Botânico (botanical gardens). Uber between the main bus station and to the airport was very affordable at <€10. Some discrepancies in the advertised amenities like no restaurant on property outside of breakfast and some of the rooms aren't yet updated.","Not the best location in and of itself. Close to public transportation, though, and within easy walking distance of the Parque Eduardo VII and Jardim Botânico (botanical gardens). Uber between the main bus station and to the airport was very affordable at <€10. Some discrepancies in the advertised amenities like no restaurant on property outside of breakfast and some of the rooms aren't yet updated."
429248,pt,"Na altura de tomar banho, não consegui ter água quente e esperei bastante tempo. Passado um tempo consegui ter água quente no lavatório.","When it was time to shower, I couldn't get hot water and waited a long time. After a while I managed to get hot water in the washbasin."
57460,es,Que no tuviera un baño en la habitación,That it didn't have a bathroom in the room.
302470,en-us,the communication with the spa was basically non-existent. Every time we went down to the spa to book a massage no one was there and it said to send an email. We sent an email and they booked us a 3pm appt. We got there at 250p and no one ever showed up. They emailed us at 320pm saying they were running 10 mins late (already 20 mins late) and then by 330 still never showed up and we had to leave so never got our massage. it was pretty unprofessional in my opinion.,the communication with the spa was basically non-existent. Every time we went down to the spa to book a massage no one was there and it said to send an email. We sent an email and they booked us a 3pm appt. We got there at 250p and no one ever showed up. They emailed us at 320pm saying they were running 10 mins late (already 20 mins late) and then by 330 still never showed up and we had to leave so never got our massage. it was pretty unprofessional in my opinion.
225138,pt-br,pequena e sem vista,Small and without a view.
397381,pt,"A apontar será mesmo o acesso ao alojamento, que na altura que viajei estava um pouco degradado, sendo uma estrada de terra batida poderia estar com uma melhor manutençao.","The only thing to note is the access to the accommodation, which when I traveled was somewhat degraded; being a dirt road it could use better maintenance."
200925,it,Arredamento troppo essenziale e un po’ datato,Furnishings are too basic and a bit dated.
246367,fr,"Le réseau d'eau, faible pression.",The water supply had low pressure.
43715,de,"Die Sauberkeit unseres Zimmers lies sehr zu wünschen übrig. Es begann gleich bei unserer Ankunft, über dem Toiletten Deckel war zwar ein Papierbogen angebracht, der die erfolgte Reinigung vermitteln sollte. Als ich den Deckel jedoch öffnete, musste ich feststellen, dass die Toilette voll Kot war. 🤮. Danach fanden wir noch ein fremdes Haar in der Dusche. Bei der Reklamation an der Rezeption wurde zwar wieder eine Reinigungsfrau ins Zimmer entsandt, jedoch ist dieser Vorfall eigentlich nicht zu tolerieren. Weiter wurde uns nur ein Zahnputzbecher ins Bad gestellt und erst bei zweimaliger Reklamation, einer zweiter gegeben. Am letzten Tag bekamen wir keine zwei kleine Handtücher mehr aufs Zimmer und mussten diese wiederum bei der Rezeption abholen.","The cleanliness of our room left much to be desired. It began right at our arrival: a paper sheet was placed over the toilet lid to indicate it had been cleaned. However, when I opened the lid I found that the toilet was full of feces. 🤮 After that we also found a foreign hair in the shower. When we complained at reception another cleaning lady was sent to the room, but this incident is really unaccep

In [11]:
error_pattern = (
    r"Error 500|Server Error|"
    r"That's an error|That’s an error|"
    r"Please try again later"
)

for col in [
    "en_review_text_liked",
    "en_review_text_disliked"
]:
    bad = (
        reviews_by_hotel[col]
        .astype("string")
        .str.contains(
            error_pattern,
            case=False,
            na=False,
            regex=True
        )
    )

    print(
        f"{col}: "
        f"{bad.sum():,} traduções suspeitas"
    )

en_review_text_liked: 0 traduções suspeitas
en_review_text_disliked: 0 traduções suspeitas


In [12]:
audit = reviews_by_hotel[
    reviews_by_hotel["review_text_disliked"].notna()
    & reviews_by_hotel["en_review_text_disliked"].notna()
].copy()

audit["original_len"] = (
    audit["review_text_disliked"]
    .astype(str)
    .str.len()
)

audit["translation_len"] = (
    audit["en_review_text_disliked"]
    .astype(str)
    .str.len()
)

audit["length_ratio"] = (
    audit["translation_len"]
    / audit["original_len"].replace(0, pd.NA)
)

display(
    audit[
        [
            "original_lang",
            "review_text_disliked",
            "en_review_text_disliked",
            "length_ratio"
        ]
    ]
    .sort_values("length_ratio")
    .head(20)
)

,original_lang,review_text_disliked,en_review_text_disliked,length_ratio
444487,pt-br,academia ginástica,gym,0.166667
285411,el,"Λόγω του ξύλινου δαπέδου ακούγεται το τρίξιμο όποτε περπατήσει κάποιος. Βέβαια, αυτό ειναι αναπόφευκτο και συνηθίζεται εύκολα Debido al suelo de madera, se puede oír el crujido cada vez que alguien camina. Por supuesto, esto es inevitable y es fácil acostumbrarse. Because of the wooden floor, you can hear the creak whenever someone walks. Of course, this is inevitable and easy to get used to","Because of the wooden floor, you can hear the creak whenever someone walks. Of course, this is inevitable and easy to get used to.",0.329949
250772,es,Las hormigas,Ants,0.333333
326447,de,Ein bisschen in die Jahre gekommen.,A bit dated.,0.342857
57867,fr,????? je me demande bien !,I wonder!,0.346154
291747,fr,Changement de serviettes de toilettes,Towel change.,0.351351
89968,pt,Estacionamento com pagamento extra,Paid parking,0.352941
54891,hu,Nincs ilyen,None,0.363636
155867,pt,alimentação,Food,0.363636
154668,pt,Alimentação,Food,0.363636


In [ ]:
reviews_by_hotel[
    [
        "original_lang",
        "review_text_liked",
        "en_review_text_liked",
        "review_text_disliked",
        "en_review_text_disliked",
        "full_review",
        "en_full_review"
    ]
].head(3).T

## Code - Dev

In [8]:
reviews_by_hotel.head(3)

,Unnamed: 0,nome_hotel_url,username,user_country,room_view,stay_duration,stay_type,review_post_date,review_title,rating,original_lang,review_text_liked,review_text_disliked,full_review,en_full_review,found_helpful,found_unhelpful,owner_resp_text,ID_HOTEL,ID_LOCAL,NUTS2,N_ROOMS,NAME,booking_url,obtained_reviews,hotel?,stars,amenities,principais_comodidades,day,month,year,en_review_text_liked,en_review_text_disliked
0,0,hotelmundial,Hiroyuki,Japan,Double or Twin Room,3 nights,Couple,04-30-2025 00:00:00,Very good,8.0,ja,レンタカーが故障した時に、レンタカー会社とポルトガル語で交渉してくれるなど親切な対応をしてくれて、とても助かりました。,None,title: Very good. liked: レンタカーが故障した時に、レンタカー会社とポルトガル語で交渉してくれるなど親切な対応をしてくれて、とても助かりました。.,None,0,0,None,149,Lisboa,Grande Lisboa,349,Hotel Mundial,https://www.booking.com/hotel/pt/hotelmundial.html,sim,yes,4.0,None,"Estacionamento privado, Acesso Wi-Fi gratuito, Transfer (aeroporto), Quartos familiares, Quartos para não fumadores, Restaurante, Receção disponível 24 horas, Comodidades para pessoas com mobilidade condicionada, Bar, Muito bom pequeno-almoço",30,4,2025,"When our rental car broke down, they kindly helped us by negotiating with the rental company in Portuguese, which was very helpful.",<NA>
1,1,hotelmundial,Helen,Hong Kong,Double or Twin Room,3 nights,Couple,04-30-2025 00:00:00,We had a very good stay and were impressed by the professionalism of the staff.,9.0,en,Location was excellent. The staff were friendly and helpful.,More variety of food at breakfast.,title: We had a very good stay and were impressed by the professionalism of the staff. liked: Location was excellent. The staff were friendly and helpful. disliked: More variety of food at breakfast.,title: We had a very good stay and were impressed by the professionalism of the staff. liked: Location was excellent. The staff were friendly and helpful. disliked: More variety of food at breakfast.,0,0,None,149,Lisboa,Grande Lisboa,349,Hotel Mundial,https://www.booking.com/hotel/pt/hotelmundial.html,sim,yes,4.0,None,"Estacionamento privado, Acesso Wi-Fi gratuito, Transfer (aeroporto), Quartos familiares, Quartos para não fumadores, Restaurante, Receção disponível 24 horas, Comodidades para pessoas com mobilidade condicionada, Bar, Muito bom pequeno-almoço",30,4,2025,Location was excellent. The staff were friendly and helpful.,More variety of food at breakfast.
2,2,hotelmundial,Uwe,Germany,Double or Twin Room,3 nights,Couple,04-30-2025 00:00:00,Kommen gerne wieder!,9.0,de,Besonders gefallen hat uns die zentrale Lage. Sehr gute Erreichbarkeit des öffentlichen Nahverkehrs. Die berühmte Tram 28 und 12 fahren nur wenige Meter vorm Hotel ab. Gleich um die Ecke sind die Haltestellen der Sightseeing Busse. Frühstück war super. Personal hilfsbereit und freundlich.,Nichts,title: Kommen gerne wieder! liked: Besonders gefallen hat uns die zentrale Lage. Sehr gute Erreichbarkeit des öffentlichen Nahverkehrs. Die berühmte Tram 28 und 12 fahren nur wenige Meter vorm Hotel ab. Gleich um die Ecke sind die Haltestellen der Sightseeing Busse. Frühstück war super. Personal hilfsbereit und freundlich. disliked: Nichts.,None,0,0,None,149,Lisboa,Grande Lisboa,349,Hotel Mundial,https://www.booking.com/hotel/pt/hotelmundial.html,sim,yes,4.0,None,"Estacionamento privado, Acesso Wi-Fi gratuito, Transfer (aeroporto), Quartos familiares, Quartos para não fumadores, Restaurante, Receção disponível 24 horas, Comodidades para pessoas com mobilidade condicionada, Bar, Muito bom pequeno-almoço",30,4,2025,We particularly liked the central location. Very good accessibility of public transport. The famous trams 28 and 12 depart just a few meters in front of the hotel. The sightseeing bus stops are just around the corner. Breakfast was great. Staff helpful and friendly.,Nichts


In [9]:
# ======================================================
# NORMALIZAÇÃO DOS NOMES DAS COLUNAS
# ======================================================

datasets_weekly = [
    df_info_hotel_ocupacao_kpis_weekly,
    dataset_sentiment_and_emotion_analysis_year_week,
    dataset_sentiment_and_emotion_analysis_year_week_artigo3,
]

for dataset in datasets_weekly:
    dataset.columns = (
        dataset.columns
        .str.strip()
        .str.upper()
    )

reviews_by_hotel.columns = reviews_by_hotel.columns.str.strip()
info_extra_hotel.columns = info_extra_hotel.columns.str.strip()


# ======================================================
# NORMALIZAÇÃO DE WEEK_YEAR
# ======================================================

df_info_hotel_ocupacao_kpis_weekly = (
    df_info_hotel_ocupacao_kpis_weekly
    .rename(columns={"YEAR_WEEK": "WEEK_YEAR"})
)


def normalize_week_year(series):
    """
    Converte formatos como:
    2024-01
    2024_01
    2024-W01
    2024_W01

    para:
    2024_W01
    """
    return (
        series.astype("string")
        .str.strip()
        .str.replace("-", "_", regex=False)
        .str.replace(r"^(\d{4})_(\d{1,2})$", r"\1_W\2", regex=True)
        .str.replace(r"^(\d{4})_W(\d)$", r"\1_W0\2", regex=True)
    )


for dataset in datasets_weekly:
    if "WEEK_YEAR" in dataset.columns:
        dataset["WEEK_YEAR"] = normalize_week_year(dataset["WEEK_YEAR"])


# Garantir que ID_HOTEL tem o mesmo tipo em todos os datasets
for dataset in datasets_weekly:
    if "ID_HOTEL" in dataset.columns:
        dataset["ID_HOTEL"] = pd.to_numeric(
            dataset["ID_HOTEL"],
            errors="coerce"
        ).astype("Int64")


reviews_by_hotel["ID_HOTEL"] = pd.to_numeric(
    reviews_by_hotel["ID_HOTEL"],
    errors="coerce"
).astype("Int64")

info_extra_hotel["ID_HOTEL"] = pd.to_numeric(
    info_extra_hotel["ID_HOTEL"],
    errors="coerce"
).astype("Int64")

df_info_hotel_ocupacao_kpis_weekly["WEEK_YEAR"] = (
    df_info_hotel_ocupacao_kpis_weekly["WEEK_YEAR"]
    .astype(str)
    .str.strip()
    .str.replace("-", "_", regex=False)
)

df_info_hotel_ocupacao_kpis_weekly["WEEK_YEAR"] = (
    df_info_hotel_ocupacao_kpis_weekly["WEEK_YEAR"]
    .str.replace(
        r"^(\d{4})_(\d{1,2})$",
        lambda m: f"{m.group(1)}_W{int(m.group(2)):02d}",
        regex=True
    )
)

In [10]:
df_info_hotel_ocupacao_kpis_weekly.tail(3)

,UNNAMED: 0,ID_HOTEL,WEEK_YEAR,QUARTOS_OCUPADOS,ID_LOCAL,NUTS2,N_ROOMS,NAME,BOOKING_URL,NOME_HOTEL_URL,OBTAINED_REVIEWS,HOTEL?,STARS,AMENITIES,PRINCIPAIS_COMODIDADES,QUARTOS_DISPONIVEIS,TAXA_OCUPACAO,N_ROOMS_OCCUPIED_WEEK,REVPAR_WEEK,REVPOR_WEEK,TREVPAR_WEEK
58775,58775,197,2021_W23,7,Leiria,Centro (PT),16,Most Art Boutique Hostel,https://www.booking.com/hotel/pt/most-art-boutique-hostel.html,most-art-boutique-hostel,sim,yes,NaN,NaN,"Estacionamento gratuito, Acesso Wi-Fi gratuito, Quartos familiares, Quartos para não fumadores, Comodidades para pessoas com mobilidade condicionada, Transfer (aeroporto), Comodidades para fazer chá e café em todos os quartos",112,6.250000,NaN,NaN,NaN,NaN
58776,58776,101,2022_W03,2,AlcobaÃ§a,Oeste e Vale do Tejo,23,Albergaria SÃ£o Pedro,https://www.booking.com/hotel/pt/albergaria-sao-pedro.html,albergaria-sao-pedro,sim,yes,NaN,NaN,"Acesso Wi-Fi gratuito, Frente à praia, Quartos familiares, Transfer (aeroporto), Serviço de quartos, Quartos para não fumadores, Bar, Excecional pequeno-almoço",161,1.242236,3.0,2.666122,40.880533,2.666122
58777,58777,122,2023_W18,6,Albufeira,Algarve,8,Balaia Plaza AL,https://www.booking.com/hotel/pt/balaia-plaza-al.html,balaia-plaza-al,sim,yes,NaN,"O espaço é todo seu, 74 m² tamanho, Cozinha, Jardim, Piscina, Máquina de lavar roupa, Acesso Wi-Fi gratuito, Terraço, Varanda, Estacionamento gratuito","Piscina exterior, Estacionamento gratuito, Acesso Wi-Fi gratuito, Quartos para não fumadores, Terraço, Jardim, Ar condicionado",56,10.714286,6.0,13.679250,109.434000,13.679250


In [11]:
# ======================================================
# REMOVER COLUNAS REPETIDAS DO DATASET DO ARTIGO 3
# ======================================================

cols_repetidas = [
    "DOMINANT_EMOTION",
    "EMOTION_ENTROPY",
    "N_REVIEWS",
    "SENTIMENT_SCORE_AVG",
    "SENTIMENT_SCORE_STD",
]

dataset_sentiment_and_emotion_analysis_year_week_artigo3 = (
    dataset_sentiment_and_emotion_analysis_year_week_artigo3
    .drop(columns=cols_repetidas, errors="ignore")
)


# ======================================================
# CONSTRUIR DATASET FINAL
# ======================================================

df_final = (
    df_info_hotel_ocupacao_kpis_weekly
    .merge(
        dataset_sentiment_and_emotion_analysis_year_week,
        on=["ID_HOTEL", "WEEK_YEAR"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        dataset_sentiment_and_emotion_analysis_year_week_artigo3,
        on=["ID_HOTEL", "WEEK_YEAR"],
        how="left",
        validate="one_to_one",
    )
)

In [12]:
df_final.count()

UNNAMED: 0_x                             58778
ID_HOTEL                                 58778
WEEK_YEAR                                58778
QUARTOS_OCUPADOS                         58778
ID_LOCAL                                 58778
NUTS2                                    58778
N_ROOMS                                  58778
NAME                                     58778
BOOKING_URL                              58778
NOME_HOTEL_URL                           58778
OBTAINED_REVIEWS                         58778
HOTEL?                                   58778
STARS                                    38113
AMENITIES                                 5973
PRINCIPAIS_COMODIDADES                   58778
QUARTOS_DISPONIVEIS                      58778
TAXA_OCUPACAO                            58778
N_ROOMS_OCCUPIED_WEEK                    39575
REVPAR_WEEK                              39539
REVPOR_WEEK                              39539
TREVPAR_WEEK                             39539
N_REVIEWS    

In [13]:

# Remover colunas criadas automaticamente na exportação dos CSV
unnamed_cols = [
    col for col in df_final.columns
    if col.upper().startswith("UNNAMED:")
]

df_final = df_final.drop(columns=unnamed_cols, errors="ignore")


# Filtrar o período de análise
df_final = df_final[
    df_final["WEEK_YEAR"].between("2023_W01", "2024_W52")
].copy()


def parse_week_year(value):
    if pd.isna(value):
        return pd.NaT

    try:
        year, week = str(value).split("_W")
        return pd.Timestamp.fromisocalendar(
            int(year),
            int(week),
            1,
        )
    except (ValueError, TypeError):
        return pd.NaT


df_final["WEEK_DATE"] = df_final["WEEK_YEAR"].apply(parse_week_year)

df_final = (
    df_final
    .dropna(subset=["ID_HOTEL", "WEEK_YEAR", "WEEK_DATE"])
    .sort_values(["ID_HOTEL", "WEEK_DATE"])
    .drop_duplicates(subset=["ID_HOTEL", "WEEK_YEAR"])
    .reset_index(drop=True)
)

print("Dimensão do painel:", df_final.shape)
print("Hotéis:", df_final["ID_HOTEL"].nunique())
print("Período:", df_final["WEEK_DATE"].min(), "a", df_final["WEEK_DATE"].max())

Dimensão do painel: (28845, 122)
Hotéis: 329
Período: 2023-01-02 00:00:00 a 2024-12-23 00:00:00


In [14]:
df_final.head(3)

,ID_HOTEL,WEEK_YEAR,QUARTOS_OCUPADOS,ID_LOCAL,NUTS2,N_ROOMS,NAME,BOOKING_URL,NOME_HOTEL_URL,OBTAINED_REVIEWS,HOTEL?,STARS,AMENITIES,PRINCIPAIS_COMODIDADES,QUARTOS_DISPONIVEIS,TAXA_OCUPACAO,N_ROOMS_OCCUPIED_WEEK,REVPAR_WEEK,REVPOR_WEEK,TREVPAR_WEEK,N_REVIEWS,AVG_RATING,RATING_STD,EXPECTATION_INTENSITY_AVG,EXPERIENCE_INTENSITY_AVG,EXPECTATION_SENTIMENT_AVG,EXPERIENCE_SENTIMENT_AVG,EXPECTATION_EXPERIENCE_GAP_AVG,SEMANTIC_ALIGNMENT_SCORE_AVG,SEMANTIC_MISALIGNMENT_SCORE_AVG,EXPECTATION_EXPERIENCE_GAP_STD,SEMANTIC_MISALIGNMENT_SCORE_STD,EXPECTATION_TOPICS_LIST,EXPERIENCE_TOPICS_LIST,VIOLATED_EXPECTATION_TOPICS_LIST,CONFIRMED_EXPECTATION_TOPICS_LIST,LATENT_EXPECTATION_TOPICS_LIST,EXPECTATION_TOPICS_UNIQUE,EXPERIENCE_TOPICS_UNIQUE,VIOLATED_EXPECTATION_TOPICS_UNIQUE,CONFIRMED_EXPECTATION_TOPICS_UNIQUE,LATENT_EXPECTATION_TOPICS_UNIQUE,N_EXPECTATION_TOPICS,N_EXPERIENCE_TOPICS,N_VIOLATED_EXPECTATION_TOPICS,N_CONFIRMED_EXPECTATION_TOPICS,N_LATENT_EXPECTATION_TOPICS,N_REVIEWS_WITH_EXPECTATION,N_REVIEWS_WITH_VIOLATED_EXPECTATION,EXPECTATION_COVERAGE,VIOLATION_COVERAGE,EXPECTATION_VIOLATION_RATIO,EXPECTATION_SHARE,HIGH_EXPECTATION_SHARE,HIGH_ALIGNMENT_SHARE,HIGH_MISALIGNMENT_SHARE,DOMINANT_EXPECTATION_EXPERIENCE_LABEL,EXPERIENCE_ONLY_SHARE,MINOR_MISALIGNMENT_SHARE,MODERATE_MISALIGNMENT_SHARE,MAJOR_MISALIGNMENT_SHARE,UNCLEAR_SHARE,EMOTION_ENTROPY,DOMINANT_EMOTION,SENTIMENT_SCORE_AVG,SENTIMENT_SCORE_STD,EXPECTATION_EXPERIENCE_GAP_AVG_LAG1,EXPECTATION_EXPERIENCE_GAP_AVG_ROLL4,SEMANTIC_MISALIGNMENT_SCORE_AVG_LAG1,SEMANTIC_MISALIGNMENT_SCORE_AVG_ROLL4,EXPECTATION_VIOLATION_RATIO_LAG1,EXPECTATION_VIOLATION_RATIO_ROLL4,MAJOR_MISALIGNMENT_SHARE_LAG1,MAJOR_MISALIGNMENT_SHARE_ROLL4,EMOTION_ENTROPY_LAG1,EMOTION_ENTROPY_ROLL4,SENTIMENT_SCORE_AVG_LAG1,SENTIMENT_SCORE_AVG_ROLL4,EXPECTATION_SHARE_LAG1,EXPECTATION_SHARE_ROLL4,HIGH_EXPECTATION_SHARE_LAG1,HIGH_EXPECTATION_SHARE_ROLL4,HIGH_ALIGNMENT_SHARE_LAG1,HIGH_ALIGNMENT_SHARE_ROLL4,HIGH_MISALIGNMENT_SHARE_LAG1,HIGH_MISALIGNMENT_SHARE_ROLL4,HIGH_MISALIGNMENT,HIGH_EXPECTATION,HIGH_EMOTIONAL_COMPLEXITY,HIGH_EXPECTATION_VIOLATION,HIGH_POSITIVE_EXPERIENCE,N_REVIEWS_POSITIVE,N_REVIEWS_NEGATIVE,PERC_REVIEWS_POSITIVE,PERC_REVIEWS_NEGATIVE,AVG_REVIEW_LENGTH_POS,AVG_REVIEW_LENGTH_NEG,TOTAL_REVIEW_LENGTH_MEAN,TOTAL_REVIEW_LENGTH_STD,%EXTREME_RATINGS_1,%EXTREME_RATINGS_10,AVG_REVIEW_RATING,MEDIAN_REVIEW_RATING,RATING_DISPERSION,NEGATIVE_TO_POSITIVE_RATIO,SEASON_FLAG,PERC_NEUTRAL_REVIEWS,N_DISTINCT_TOPICS,TOPIC_CONCENTRATION_INDEX,PERC_OPERATIONAL_COMPLAINTS,PERC_STRUCTURAL_PRAISE,PERC_PRICE_VALUE_MENTIONS,EMERGING_NEGATIVE_TOPIC,POLARIZATION_INDEX,PRIORITY_ALERT_FLAG,REVIEWER_ORIGIN_DIVERSITY,COMPLEXITY_INDEX,DELTA_SENTIMENT_VS_PREV_WEEK,DELTA_NUM_REVIEWS_VS_PREV_WEEK,ROLLING_AVG_SENTIMENT_4W,SENTIMENT_TREND_SLOPE,WEEK_DATE
0,1,2023_W01,461,Lagoa,Algarve,126,Agua Hotels Riverside Hotel Apartamento,https://www.booking.com/hotel/pt/agua-hotels-riverside.html,agua-hotels-riverside,sim,yes,4.0,NaN,"2 piscinas, Acesso Wi-Fi gratuito, Estacionamento gratuito, Spa e centro de bem-estar, Centro de fitness, Comodidades para pessoas com mobilidade condicionada, Transfer (aeroporto), Quartos para não fumadores, Bar, Muito bom pequeno-almoço",882,52.267574,447.0,17.718202,32.159364,30.408017,15.0,8.6,1.502379,0.050000,0.266667,-0.50,0.450000,0.166667,0.825,0.175,0.275076,0.292831,"['towel/pool deposit process should be linked to the hotel room (no paper ticket required)', ""New Year's Eve dinner quality relative to price"", ""coffee included with the New Year's Eve dinner""]","['breakfast variety', 'location', 'river view', 'room quality (excellent, spacious, modern)', 'breakfast quality and variety (complete, many options)', 'sofa comfort (sofa in living room not comfortable)', 'kitchen equipment (missing several essential utensils)', 'reception assistance (staff provided missing utensils)', 'towel/pool deposit process (had to keep a paper ticket to reclaim deposit)', 'outdoor space', 'value for money', 'daily cleaning', 'bathro

In [15]:
# ======================================================
# PREPARAR REVIEWS
# ======================================================

reviews_clean = reviews_by_hotel.copy()

reviews_clean = reviews_clean.drop(
    columns=[
        col for col in reviews_clean.columns
        if col.lower().startswith("unnamed:")
    ],
    errors="ignore",
)


# Detetar a coluna da data
possible_date_columns = [
    "review_post_date",
    "REVIEW_POST_DATE",
    "review_date",
    "REVIEW_DATE",
]

review_date_column = next(
    (
        col for col in possible_date_columns
        if col in reviews_clean.columns
    ),
    None,
)

if review_date_column is None:
    raise KeyError(
        "Não foi encontrada uma coluna de data das reviews. "
        f"Colunas disponíveis: {reviews_clean.columns.tolist()}"
    )


# Não restringir a um único formato, para suportar pequenas variações
reviews_clean["review_date"] = pd.to_datetime(
    reviews_clean[review_date_column],
    errors="coerce",
)


# Segunda tentativa para o formato original, caso necessário
invalid_dates = reviews_clean["review_date"].isna()

if invalid_dates.any():
    reviews_clean.loc[invalid_dates, "review_date"] = pd.to_datetime(
        reviews_clean.loc[invalid_dates, review_date_column],
        format="%m-%d-%Y %H:%M:%S",
        errors="coerce",
    )


reviews_clean["week_date"] = (
    reviews_clean["review_date"]
    - pd.to_timedelta(
        reviews_clean["review_date"].dt.weekday,
        unit="D",
    )
).dt.normalize()


iso_calendar = reviews_clean["review_date"].dt.isocalendar()

reviews_clean["WEEK_YEAR"] = (
    iso_calendar["year"].astype("Int64").astype("string")
    + "_W"
    + iso_calendar["week"].astype("Int64").astype("string").str.zfill(2)
)


reviews_clean["ID_HOTEL"] = pd.to_numeric(
    reviews_clean["ID_HOTEL"],
    errors="coerce",
).astype("Int64")


print("Reviews totais:", len(reviews_clean))
print("Datas inválidas:", reviews_clean["review_date"].isna().sum())
print("Data mínima:", reviews_clean["review_date"].min())
print("Data máxima:", reviews_clean["review_date"].max())
print("Hotéis:", reviews_clean["ID_HOTEL"].nunique())

Reviews totais: 452061
Datas inválidas: 0
Data mínima: 2022-04-30 00:00:00
Data máxima: 2025-05-02 00:00:00
Hotéis: 333


In [16]:
# ======================================================
# LIMITAR REVIEWS AO PAINEL
# ======================================================

df_final_clean = (
    df_final
    .drop_duplicates(subset=["ID_HOTEL", "WEEK_YEAR"])
    .copy()
)

df_final_clean["WEEK_DATE"] = pd.to_datetime(
    df_final_clean["WEEK_DATE"],
    errors="coerce",
).dt.normalize()


panel_start = df_final_clean["WEEK_DATE"].min()
panel_end = df_final_clean["WEEK_DATE"].max() + pd.Timedelta(days=6)


reviews_panel_period = reviews_clean[
    reviews_clean["review_date"].between(
        panel_start,
        panel_end,
        inclusive="both",
    )
].copy()


hotels_panel = set(
    df_final_clean["ID_HOTEL"]
    .dropna()
    .astype(int)
)

hotels_reviews = set(
    reviews_panel_period["ID_HOTEL"]
    .dropna()
    .astype(int)
)

common_hotels = sorted(hotels_panel & hotels_reviews)
only_reviews = sorted(hotels_reviews - hotels_panel)
only_panel = sorted(hotels_panel - hotels_reviews)


reviews_panel_period = reviews_panel_period[
    reviews_panel_period["ID_HOTEL"].isin(common_hotels)
].copy()

df_final_clean = df_final_clean[
    df_final_clean["ID_HOTEL"].isin(common_hotels)
].copy()


print("Início do painel:", panel_start)
print("Fim do painel:", panel_end)
print("Reviews dentro do período:", len(reviews_panel_period))
print("Hotéis comuns:", len(common_hotels))
print("Apenas nas reviews:", only_reviews)
print("Apenas no painel:", only_panel)
print("Dimensão final do painel:", df_final_clean.shape)

Início do painel: 2023-01-02 00:00:00
Fim do painel: 2024-12-29 00:00:00
Reviews dentro do período: 301318
Hotéis comuns: 329
Apenas nas reviews: [273, 307, 510, 547]
Apenas no painel: []
Dimensão final do painel: (28845, 122)


In [17]:
# ======================================================
# VALIDAÇÕES
# ======================================================

print("Painel:")
print(df_final_clean.shape)
print(df_final_clean[["ID_HOTEL", "WEEK_YEAR", "WEEK_DATE"]].head())

print("\nReviews:")
print(reviews_panel_period.shape)
print(
    reviews_panel_period[
        ["ID_HOTEL", "WEEK_YEAR", "review_date"]
    ].head()
)

print("\nDuplicados no painel:")
print(
    df_final_clean
    .duplicated(["ID_HOTEL", "WEEK_YEAR"])
    .sum()
)

print("\nReviews sem ID_HOTEL:")
print(reviews_panel_period["ID_HOTEL"].isna().sum())

print("\nReviews sem data:")
print(reviews_panel_period["review_date"].isna().sum())

Painel:
(28845, 122)
   ID_HOTEL WEEK_YEAR  WEEK_DATE
0         1  2023_W01 2023-01-02
1         1  2023_W02 2023-01-09
2         1  2023_W03 2023-01-16
3         1  2023_W04 2023-01-23
4         1  2023_W05 2023-01-30

Reviews:
(301318, 36)
      ID_HOTEL WEEK_YEAR review_date
1205       149  2024_W52  2024-12-29
1206       149  2024_W52  2024-12-29
1207       149  2024_W52  2024-12-29
1208       149  2024_W52  2024-12-29
1209       149  2024_W52  2024-12-29

Duplicados no painel:
0

Reviews sem ID_HOTEL:
0

Reviews sem data:
0


In [18]:
# ======================================================
# PREPARAR DATASETS PARA O CASE BUILDER
# ======================================================

panel_for_case_builder = df_final_clean.copy()
reviews_for_case_builder = reviews_panel_period.copy()


# O CaseBuilder espera "week_date" em minúsculas
if "WEEK_DATE" in panel_for_case_builder.columns:
    panel_for_case_builder = panel_for_case_builder.rename(
        columns={"WEEK_DATE": "week_date"}
    )


# Garantir tipos corretos
panel_for_case_builder["ID_HOTEL"] = pd.to_numeric(
    panel_for_case_builder["ID_HOTEL"],
    errors="coerce"
).astype("Int64")

panel_for_case_builder["week_date"] = pd.to_datetime(
    panel_for_case_builder["week_date"],
    errors="coerce"
).dt.normalize()


# O CaseBuilder espera também WEEK_YEAR
panel_for_case_builder["WEEK_YEAR"] = (
    panel_for_case_builder["WEEK_YEAR"]
    .astype("string")
    .str.strip()
)


# Remover linhas inválidas
panel_for_case_builder = (
    panel_for_case_builder
    .dropna(
        subset=[
            "ID_HOTEL",
            "WEEK_YEAR",
            "week_date",
        ]
    )
    .sort_values(
        ["ID_HOTEL", "week_date"]
    )
    .drop_duplicates(
        subset=["ID_HOTEL", "week_date"],
        keep="first"
    )
    .reset_index(drop=True)
)


# Reviews
reviews_for_case_builder["ID_HOTEL"] = pd.to_numeric(
    reviews_for_case_builder["ID_HOTEL"],
    errors="coerce"
).astype("Int64")


# O CaseBuilder volta a criar review_date internamente
# a partir de review_post_date, por isso esta coluna tem de existir
if "review_post_date" not in reviews_for_case_builder.columns:
    raise KeyError(
        "A coluna 'review_post_date' não existe nas reviews."
    )


reviews_for_case_builder = (
    reviews_for_case_builder
    .dropna(subset=["ID_HOTEL", "review_post_date"])
    .reset_index(drop=True)
)


print("Painel preparado:")
print(panel_for_case_builder.shape)
print(
    panel_for_case_builder[
        ["ID_HOTEL", "WEEK_YEAR", "week_date"]
    ].head()
)

print("\nReviews preparadas:")
print(reviews_for_case_builder.shape)
print(
    reviews_for_case_builder[
        ["ID_HOTEL", "review_post_date"]
    ].head()
)

print(
    "\nDuplicados ID_HOTEL + week_date:",
    panel_for_case_builder
    .duplicated(["ID_HOTEL", "week_date"])
    .sum()
)

Painel preparado:
(28845, 122)
   ID_HOTEL WEEK_YEAR  week_date
0         1  2023_W01 2023-01-02
1         1  2023_W02 2023-01-09
2         1  2023_W03 2023-01-16
3         1  2023_W04 2023-01-23
4         1  2023_W05 2023-01-30

Reviews preparadas:
(301318, 36)
   ID_HOTEL     review_post_date
0       149  12-29-2024 00:00:00
1       149  12-29-2024 00:00:00
2       149  12-29-2024 00:00:00
3       149  12-29-2024 00:00:00
4       149  12-29-2024 00:00:00

Duplicados ID_HOTEL + week_date: 0


In [19]:
from src.case_builder import CaseBuilder

In [20]:
builder = CaseBuilder(
    panel_df=panel_for_case_builder,
    reviews_df=reviews_for_case_builder
)

In [21]:
case = builder.build_case(
    hotel_id=149,
    decision_week="2024_W20",
    lookback_weeks=4,
    max_reviews=20,
    top_n_topics=10
)

In [22]:
complaints_df = pd.DataFrame(
    case["complaints"]
)

display(
    complaints_df[
        [
            "review_date",
            "rating",
            "language",
            "complaint_text",
            "disliked",
            "liked_context"
        ]
    ]
)

,review_date,rating,language,complaint_text,disliked,liked_context
0,2024-04-26,5.0,en-us,"Disliked: Reflecting on my stay at Hotel Mundial, there were several aspects that left me feeling underwhelmed. Firstly, the room and bathroom fell short of my expectations, with outdated furnishings and a cramped layout. The shower posed a safety concern due to its slippery floor and constant leakage, making it difficult to use comfortably. Additionally, the bedding arrangement, with two twin beds pushed together instead of a king bed as requested, was disappointing. The lack of amenities such as a steamer or iron further detracted from the overall comfort of the room. Despite the hotel's convenient location, the outdated facilities and lackluster amenities made it difficult for me to overlook these shortcomings. Overall, while there were some positive aspects to my stay, I found the hotel to be lacking in terms of value and comfort.","Reflecting on my stay at Hotel Mundial, there were several aspects that left me feeling underwhelmed. Firstly, the room and bathroom fell short of my expectations, with outdated furnishings and a cramped layout. The shower posed a safety concern due to its slippery floor and constant leakage, making it difficult to use comfortably. Additionally, the bedding arrangement, with two twin beds pushed together instead of a king bed as requested, was disappointing. The lack of amenities such as a steamer or iron further detracted from the overall comfort of the room. Despite the hotel's convenient location, the outdated facilities and lackluster amenities made it difficult for me to overlook these shortcomings. Overall, while there were some positive aspects to my stay, I found the hotel to be lacking in terms of value and comfort.","In terms of what I liked about Hotel Mundial, there were definitely some positives to highlight amidst the mixed experience. Firstly, the hotel's prime location stood out to me as a major advantage. Being close to many of Lisbon's main attractions and cities made exploring the area incredibly convenient. Additionally, the restaurant offered a picturesque view, and I particularly enjoyed the octopus salad we had for dinner. The abundance of breakfast choices was also a plus, even though I wished the quality of the food matched the variety. Finally, the staff were friendly and accommodating, which added a welcoming touch to our stay."
1,2024-05-08,6.0,en-us,Disliked: Not value for money. For the money you pay you'd expect at least a free bottle of water in the room. Breakfast is below average. Hotel is old and is a bit outdated. Doesn't have enough facilities (ex: Gym),Not value for money. For the money you pay you'd expect at least a free bottle of water in the room. Breakfast is below average. Hotel is old and is a bit outdated. Doesn't have enough facilities (ex: Gym),Location is very central. Beds are comfortable and shower was good. Staff were great and very helpful!
2,2024-04-30,6.0,de,Disliked: A hotel that has seen better days and is in urgent need of renovation. The rooms are worn and you simply don't feel comfortable. Cleanliness also leaves much to be desired. Access to the beautiful rooftop terrace should always be guaranteed for hotel guests.,A hotel that has seen better days and is in urgent need of renovation. The rooms are worn and you simply don't feel comfortable. Cleanliness also leaves much to be desired. Access to the beautiful rooftop terrace should always be guaranteed for hotel guests.,The hotel stands out for its location and the beautiful rooftop terrace.
3,2024-04-23,6.0,de,"Disliked: Stairs made of black stone. The edge is barely visible… potentially dangerous! It should be painted a light color. Large, many guests, breakfast has a high noise level, crowding 😰 The reception is often hectic as well.","Stairs made of black stone. The edge is barely visible… potentially dangerous! It should be painted a light color. Large, many guests, breakfast has a high noi

In [23]:
case

{'case_metadata': {'case_id': 'H149_20240513_L4',
  'hotel_id': 149,
  'decision_week': '2024_W20',
  'decision_date': '2024-05-13',
  'window_start': '2024-04-22',
  'window_end': '2024-05-19',
  'lookback_weeks_requested': 4,
  'panel_weeks_available': 4,
  'complete_window': True},
 'hotel_profile': {'ID_HOTEL': 149,
  'NAME': 'Hotel Mundial',
  'NOME_HOTEL_URL': 'hotelmundial',
  'ID_LOCAL': 'Lisboa',
  'N_ROOMS': 349,
  'STARS': 4.0,
  'PRINCIPAIS_COMODIDADES': 'Estacionamento privado, Acesso Wi-Fi gratuito, Transfer (aeroporto), Quartos familiares, Quartos para não fumadores, Restaurante, Receção disponível 24 horas, Comodidades para pessoas com mobilidade condicionada, Bar, Muito bom pequeno-almoço'},
 'latest_context': {'dominant_emotion': 'joy',
  'dominant_expectation_experience_label': 'experience_only',
  'season': 'Low',
  'priority_alert_flag': 0.0,
  'emerging_negative_topic': ['overpriced',
   'shower experience',
   'toilet comfort',
   'space',
   'no showers',
   'co

In [24]:
complaints_df = pd.DataFrame(case["complaints"])

display(
    complaints_df[
        [
            "review_date",
            "rating",
            "language",
            "title",
            "disliked",
            "complaint_text"
        ]
    ]
)

,review_date,rating,language,title,disliked,complaint_text
0,2024-04-26,5.0,en-us,Convenient location and friendly staff contrasted by outdated facilities and disappointing room amen,"Reflecting on my stay at Hotel Mundial, there were several aspects that left me feeling underwhelmed. Firstly, the room and bathroom fell short of my expectations, with outdated furnishings and a cramped layout. The shower posed a safety concern due to its slippery floor and constant leakage, making it difficult to use comfortably. Additionally, the bedding arrangement, with two twin beds pushed together instead of a king bed as requested, was disappointing. The lack of amenities such as a steamer or iron further detracted from the overall comfort of the room. Despite the hotel's convenient location, the outdated facilities and lackluster amenities made it difficult for me to overlook these shortcomings. Overall, while there were some positive aspects to my stay, I found the hotel to be lacking in terms of value and comfort.","Disliked: Reflecting on my stay at Hotel Mundial, there were several aspects that left me feeling underwhelmed. Firstly, the room and bathroom fell short of my expectations, with outdated furnishings and a cramped layout. The shower posed a safety concern due to its slippery floor and constant leakage, making it difficult to use comfortably. Additionally, the bedding arrangement, with two twin beds pushed together instead of a king bed as requested, was disappointing. The lack of amenities such as a steamer or iron further detracted from the overall comfort of the room. Despite the hotel's convenient location, the outdated facilities and lackluster amenities made it difficult for me to overlook these shortcomings. Overall, while there were some positive aspects to my stay, I found the hotel to be lacking in terms of value and comfort."
1,2024-05-08,6.0,en-us,Pleasant,Not value for money. For the money you pay you'd expect at least a free bottle of water in the room. Breakfast is below average. Hotel is old and is a bit outdated. Doesn't have enough facilities (ex: Gym),Disliked: Not value for money. For the money you pay you'd expect at least a free bottle of water in the room. Breakfast is below average. Hotel is old and is a bit outdated. Doesn't have enough facilities (ex: Gym)
2,2024-04-30,6.0,de,Ich würde das Hotel nicht wieder buchen,A hotel that has seen better days and is in urgent need of renovation. The rooms are worn and you simply don't feel comfortable. Cleanliness also leaves much to be desired. Access to the beautiful rooftop terrace should always be guaranteed for hotel guests.,Disliked: A hotel that has seen better days and is in urgent need of renovation. The rooms are worn and you simply don't feel comfortable. Cleanliness also leaves much to be desired. Access to the beautiful rooftop terrace should always be guaranteed for hotel guests.
3,2024-04-23,6.0,de,Pleasant,"Stairs made of black stone. The edge is barely visible… potentially dangerous! It should be painted a light color. Large, many guests, breakfast has a high noise level, crowding 😰 The reception is often hectic as well.","Disliked: Stairs made of black stone. The edge is barely visible… potentially dangerous! It should be painted a light color. Large, many guests, breakfast has a high noise level, crowding 😰 The reception is often hectic as well."
4,2024-05-12,6.0,da,Pleasant,It was too expensive for what you got; it should probably be a 3-star hotel if you only looked at the rooms.,Disliked: It was too expensive for what you got; it should probably be a 3-star hotel if you only looked at the rooms.
5,2024-04-29,6.0,en,More training or more bet staff,Wait time with bar staff sometimes considerably long in the lounge bar,Disliked: Wait time with bar staff sometimes considerably long in the lounge bar
6,2024-05-19,7.0,nl,Comfortabel verblijf voor een stedentrip.,The bathroom is dated. If you're a bit taller you cannot sit properly on the toilet

In [25]:
case["trends"]["EXPECTATION_VIOLATION_RATIO"]

{'direction': 'increasing',
 'trend_strength': 'weak',
 'first_value': 0.9230769230769232,
 'last_value': 1.0,
 'minimum_value': 0.8333333333333334,
 'maximum_value': 1.0,
 'absolute_change': 0.07692307692307676,
 'percentage_change': 8.333333333333314,
 'slope': 0.020695970695970567,
 'standardized_slope': 0.3196522241219279,
 'direction_consistency': 0.3333333333333333,
 'recent_movement': 'up',
 'recent_absolute_change': 0.16666666666666663,
 'recent_reversal': False,
 'n_observations': 4}

In [26]:
pd.DataFrame(case["preliminary_signals"])

,signal,severity,metric,description,trend_strength,direction_consistency,recent_movement,recent_reversal
0,declining_sentiment,medium,SENTIMENT_SCORE_AVG,O sentimento médio apresenta uma tendência global decrescente.,moderate,0.666667,up,True
1,increasing_expectation_gap,high,EXPECTATION_EXPERIENCE_GAP_AVG,A discrepância entre expectativas e experiência apresenta uma tendência global crescente.,moderate,0.666667,down,True
2,increasing_misalignment,high,SEMANTIC_MISALIGNMENT_SCORE_AVG,O desalinhamento semântico apresenta uma tendência global crescente.,moderate,0.666667,down,True
3,increasing_emotional_complexity,medium,EMOTION_ENTROPY,A heterogeneidade emocional apresenta uma tendência global crescente.,moderate,0.666667,down,True


In [27]:
case.keys()

dict_keys(['case_metadata', 'hotel_profile', 'latest_context', 'metric_definitions', 'weekly_history', 'trends', 'topic_summary', 'preliminary_signals', 'complaints', 'complaint_metadata'])

In [28]:
case["case_metadata"]

{'case_id': 'H149_20240513_L4',
 'hotel_id': 149,
 'decision_week': '2024_W20',
 'decision_date': '2024-05-13',
 'window_start': '2024-04-22',
 'window_end': '2024-05-19',
 'lookback_weeks_requested': 4,
 'panel_weeks_available': 4,
 'complete_window': True}

In [29]:
pd.DataFrame(case["weekly_history"])

,WEEK_YEAR,week_date,REVPAR_WEEK,REVPOR_WEEK,TREVPAR_WEEK,TAXA_OCUPACAO,N_REVIEWS,AVG_RATING,SENTIMENT_SCORE_AVG,SENTIMENT_SCORE_STD,EXPECTATION_EXPERIENCE_GAP_AVG,SEMANTIC_MISALIGNMENT_SCORE_AVG,EXPECTATION_VIOLATION_RATIO,MAJOR_MISALIGNMENT_SHARE,EMOTION_ENTROPY,EXPECTATION_SHARE,HIGH_EXPECTATION_SHARE,HIGH_MISALIGNMENT_SHARE,PERC_REVIEWS_NEGATIVE,PERC_OPERATIONAL_COMPLAINTS,PERC_PRICE_VALUE_MENTIONS,POLARIZATION_INDEX,COMPLEXITY_INDEX,SENTIMENT_TREND_SLOPE
0,2024_W17,2024-04-22,137.790886,211.502883,183.412690,89.111748,47.0,8.808511,0.348485,0.621473,0.079167,0.083333,0.923077,0.000000,1.339716,0.170213,0.042553,0.063830,63.829787,14.285714,0.0,0.659574,2.564026,0.000471
1,2024_W18,2024-04-29,138.889038,298.690364,184.108465,89.070815,41.0,8.585366,0.277778,0.608608,0.120370,0.135185,0.857143,0.000000,1.385725,0.170732,0.048780,0.097561,43.902439,26.666667,0.0,0.585366,2.746082,0.000471
2,2024_W19,2024-05-06,152.305982,248.980277,197.277034,84.076955,31.0,8.612903,0.175000,0.615995,0.234615,0.250000,0.833333,0.064516,1.465536,0.193548,0.129032,0.129032,41.935484,18.181818,0.0,0.612903,2.847243,0.000471
3,2024_W20,2024-05-13,154.482965,261.135678,209.317863,87.392550,40.0,8.575000,0.285106,0.605044,0.124074,0.144444,1.000000,0.000000,1.404892,0.200000,0.000000,0.100000,50.000000,17.647059,0.0,0.500000,2.611646,0.000471


In [30]:
trend_table = (
    pd.DataFrame(case["trends"])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)

display(trend_table)

,metric,direction,trend_strength,first_value,last_value,minimum_value,maximum_value,absolute_change,percentage_change,slope,standardized_slope,direction_consistency,recent_movement,recent_absolute_change,recent_reversal,n_observations
0,REVPAR_WEEK,increasing,consistent,137.790886,154.482965,137.790886,154.482965,16.692079,12.114066,6.349318,0.838032,1.0,up,2.176983,False,4
1,REVPOR_WEEK,increasing,moderate,211.502883,261.135678,211.502883,298.690364,49.632794,23.466722,9.91883,0.318692,0.666667,up,12.155401,False,4
2,TREVPAR_WEEK,increasing,consistent,183.41269,209.317863,183.41269,209.317863,25.905173,14.123981,9.088409,0.852684,1.0,up,12.040829,False,4
3,TAXA_OCUPACAO,decreasing,moderate,89.111748,87.39255,84.076955,89.111748,-1.719198,-1.92926,-1.015145,-0.495876,0.666667,up,3.315596,True,4
4,N_REVIEWS,decreasing,moderate,47.0,40.0,31.0,47.0,-7.0,-14.893617,-3.1,-0.542214,0.666667,up,9.0,True,4
5,AVG_RATING,decreasing,moderate,8.808511,8.575,8.575,8.808511,-0.233511,-2.650966,-0.067299,-0.707227,0.666667,down,-0.037903,False,4
6,SENTIMENT_SCORE_AVG,decreasing,moderate,0.348485,0.285106,0.175,0.348485,-0.063378,-18.186864,-0.029291,-0.471103,0.666667,up,0.110106,True,4
7,SENTIMENT_SCORE_STD,decreasing,moderate,0.621473,0.605044,0.605044,0.621473,-0.016429,-2.643598,-0.00419,-0.656069,0.666667,down,-0.010952,False,4
8,EXPECTATION_EXPERIENCE_GAP_AVG,increasing,moderate,0.079167,0.124074,0.079167,0.234615,0.044907,56.725146,0.024897,0.431911,0.666667,down,-0.110541,True,4
9,SEMANTIC_MISALIGNMENT_SCORE_AVG,increasing,moderate,0.083333,0.144444,0.083333,0.25,0.061111,73.333333,0.029815,0.492609,0.666667,down,-0.105556,True,4


In [31]:
display(
    trend_table[
        trend_table["direction"].isin(
            ["increasing", "decreasing"]
        )
    ][
        [
            "metric",
            "direction",
            "first_value",
            "last_value",
            "absolute_change",
            "percentage_change",
            "standardized_slope"
        ]
    ]
)

,metric,direction,first_value,last_value,absolute_change,percentage_change,standardized_slope
0,REVPAR_WEEK,increasing,137.790886,154.482965,16.692079,12.114066,0.838032
1,REVPOR_WEEK,increasing,211.502883,261.135678,49.632794,23.466722,0.318692
2,TREVPAR_WEEK,increasing,183.41269,209.317863,25.905173,14.123981,0.852684
3,TAXA_OCUPACAO,decreasing,89.111748,87.39255,-1.719198,-1.92926,-0.495876
4,N_REVIEWS,decreasing,47.0,40.0,-7.0,-14.893617,-0.542214
5,AVG_RATING,decreasing,8.808511,8.575,-0.233511,-2.650966,-0.707227
6,SENTIMENT_SCORE_AVG,decreasing,0.348485,0.285106,-0.063378,-18.186864,-0.471103
7,SENTIMENT_SCORE_STD,decreasing,0.621473,0.605044,-0.016429,-2.643598,-0.656069
8,EXPECTATION_EXPERIENCE_GAP_AVG,increasing,0.079167,0.124074,0.044907,56.725146,0.431911
9,SEMANTIC_MISALIGNMENT_SCORE_AVG,increasing,0.083333,0.144444,0.061111,73.333333,0.492609


In [32]:
case["topic_summary"]

{'raw_weekly_topics': {'EXPECTATION_TOPICS_UNIQUE': [{'topic': 'ability to reserve parking',
    'weeks_present': 1},
   {'topic': 'ability to reserve tables', 'weeks_present': 1},
   {'topic': 'ability to use rooftop terrace without mandatory gastronomy use',
    'weeks_present': 1},
   {'topic': 'access to rooftop terrace', 'weeks_present': 1},
   {'topic': 'adequate room cleanliness / regular housekeeping',
    'weeks_present': 1},
   {'topic': 'availability of basic in-room amenities (steamer/iron)',
    'weeks_present': 1},
   {'topic': 'availability of basic toiletries at hotel (hair conditioner, body lotion)',
    'weeks_present': 1},
   {'topic': 'availability of emergency feminine hygiene products at hotel (tampons, pads)',
    'weeks_present': 1},
   {'topic': 'bathroom condition appropriate for the price / value',
    'weeks_present': 1},
   {'topic': 'bathroom renovation / improved bathroom condition',
    'weeks_present': 1}],
  'EXPERIENCE_TOPICS_UNIQUE': [{'topic': 'brea

In [33]:
case["latest_context"]

{'dominant_emotion': 'joy',
 'dominant_expectation_experience_label': 'experience_only',
 'season': 'Low',
 'priority_alert_flag': 0.0,
 'emerging_negative_topic': ['overpriced',
  'shower experience',
  'toilet comfort',
  'space',
  'no showers',
  'cocktails']}

In [34]:
case["trends"]["EXPECTATION_EXPERIENCE_GAP_AVG"]

{'direction': 'increasing',
 'trend_strength': 'moderate',
 'first_value': 0.0791666666666666,
 'last_value': 0.124074074074074,
 'minimum_value': 0.0791666666666666,
 'maximum_value': 0.2346153846153846,
 'absolute_change': 0.04490740740740741,
 'percentage_change': 56.725146198830465,
 'slope': 0.024896723646723632,
 'standardized_slope': 0.4319111355690588,
 'direction_consistency': 0.6666666666666666,
 'recent_movement': 'down',
 'recent_absolute_change': -0.11054131054131058,
 'recent_reversal': True,
 'n_observations': 4}

# AGENT CONTEXT

In [35]:
from src.agent_context import (
    build_agent_context
)

In [36]:
agent_context = build_agent_context(
    case,
    top_n_categories=8,
    max_complaints=12
)

In [37]:
agent_context.keys()

dict_keys(['context_metadata', 'case', 'hotel', 'latest_context', 'business_performance', 'experience_indicators', 'preliminary_signals', 'managerial_evidence', 'diagnostic_vulnerabilities', 'strengths', 'complaint_evidence', 'complaint_sample_metadata', 'interpretation_rules'])

In [38]:
pd.DataFrame(
    agent_context[
        "diagnostic_vulnerabilities"
    ]
)

,area,managerial_domain,default_priority,review_mentions_total,positive_mentions,negative_mentions,net_mentions,positive_share,negative_share,experience_weeks,expectation_weeks,violated_expectation_weeks,emerging_negative_weeks,has_expectation_violation,is_emerging_negative,evidence_pattern,matched_topics,source_columns,diagnostic_score,diagnostic_score_components
0,Price–value,pricing_and_value,high,18,6,12,-6,0.333,0.667,0,1,1,1,True,True,net_negative,"[acceptable value/comfort for the hotel, bathroom condition appropriate for the price / value, overpriced, parking cost]","[EMERGING_NEGATIVE_TOPIC, EXPECTATION_TOPICS_UNIQUE, LATENT_EXPECTATION_TOPICS_UNIQUE, VIOLATED_EXPECTATION_TOPICS_UNIQUE]",24,"{'negative_frequency': 10, 'net_negative': 3, 'negative_share': 2, 'expectation_violation': 4, 'emerging_negative': 3, 'default_priority': 2}"
1,Bathroom condition,rooms_and_maintenance,high,14,5,9,-4,0.357,0.643,0,1,1,0,True,False,net_negative,"[availability of basic toiletries at hotel (hair conditioner, body lotion), bathroom condition appropriate for the price / value, bathroom renovation / improved bathroom condition, bathroom storage / shelves, bathroom usability and safety (non-slip surfaces, adequate shower height, comfortable toilet for taller guests)]","[EXPECTATION_TOPICS_UNIQUE, LATENT_EXPECTATION_TOPICS_UNIQUE, VIOLATED_EXPECTATION_TOPICS_UNIQUE]",20,"{'negative_frequency': 9, 'net_negative': 3, 'negative_share': 2, 'expectation_violation': 4, 'emerging_negative': 0, 'default_priority': 2}"
2,Restaurant and bar,food_and_beverage,medium,29,20,13,7,0.606,0.394,4,1,1,1,True,True,positive_with_friction,"[ability to reserve tables, access to rooftop bar/restaurant, bar/restaurant access, rooftop bar]","[EMERGING_NEGATIVE_TOPIC, EXPECTATION_TOPICS_UNIQUE, EXPERIENCE_TOPICS_UNIQUE, LATENT_EXPECTATION_TOPICS_UNIQUE, VIOLATED_EXPECTATION_TOPICS_UNIQUE]",17,"{'negative_frequency': 10, 'net_negative': 0, 'negative_share': 0, 'expectation_violation': 4, 'emerging_negative': 3, 'default_priority': 0}"
3,Room comfort,rooms_and_maintenance,high,19,8,11,-3,0.421,0.579,4,0,0,0,False,False,net_negative,"[bed comfort, room comfort, room size]",[EXPERIENCE_TOPICS_UNIQUE],16,"{'negative_frequency': 10, 'net_negative': 3, 'negative_share': 1, 'expectation_violation': 0, 'emerging_negative': 0, 'default_priority': 2}"
4,Rooftop access,access_and_facilities,medium,29,24,8,16,0.750,0.250,4,1,1,1,True,True,positive_with_friction,"[ability to use rooftop terrace without mandatory gastronomy use, access to rooftop bar/restaurant, access to rooftop terrace, amenities advertised/featured (rooftop) usable without extra purchase, difficulties with rooftop reservations, rooftop bar]","[EMERGING_NEGATIVE_TOPIC, EXPECTATION_TOPICS_UNIQUE, EXPERIENCE_TOPICS_UNIQUE, LATENT_EXPECTATION_TOPICS_UNIQUE, VIOLATED_EXPECTATION_TOPICS_UNIQUE]",15,"{'negative_frequency': 8, 'net_negative': 0, 'negative_share': 0, 'expectation_violation': 4, 'emerging_negative': 3, 'default_priority': 0}"
5,Cleanliness and housekeeping,housekeeping,high,18,14,5,9,0.737,0.263,4,1,1,0,True,False,positive_with_friction,"[adequate room cleanliness / regular housekeeping, cleanliness]","[EXPECTATION_TOPICS_UNIQUE, EXPERIENCE_TOPICS_UNIQUE, LATENT_EXPECTATION_TOPICS_UNIQUE, VIOLATED_EXPECTATION_TOPICS_UNIQUE]",11,"{'negative_frequency': 5, 'net_negative': 0, 'negative_share': 0, 'expectation_violation': 4, 'emerging_negative': 0, 'default_priority': 2}"


In [39]:
pd.DataFrame(
    agent_context[
        "strengths"
    ]
)

,area,positive_mentions,negative_mentions,positive_share,strength_type,has_expectation_violation,is_emerging_negative
0,Staff and service,46,8,0.852,clear_strength,False,False
1,Breakfast,34,16,0.680,clear_strength,False,False
2,Rooftop access,24,8,0.750,strength_with_friction,True,True
3,Cleanliness and housekeeping,14,5,0.737,strength_with_friction,True,False


In [40]:
pd.DataFrame(
    agent_context[
        "complaint_evidence"
    ]
)

,review_date,rating,original_language,complaint,positive_context,country,stay_type
0,2024-04-26,5.0,en-us,"Disliked: Reflecting on my stay at Hotel Mundial, there were several aspects that left me feeling underwhelmed. Firstly, the room and bathroom fell short of my expectations, with outdated furnishings and a cramped layout. The shower posed a safety concern due to its slippery floor and constant leakage, making it difficult to use comfortably. Additionally, the bedding arrangement, with two twin beds pushed together instead of a king bed as requested, was disappointing. The lack of amenities such as a steamer or iron further detracted from the overall comfort of the room. Despite the hotel's convenient location, the outdated facilities and lackluster amenities made it difficult for me to overlook these shortcomings. Overall, while there were some positive aspects to my stay, I found the hotel to be lacking in terms of value and comfort.","In terms of what I liked about Hotel Mundial, there were definitely some positives to highlight amidst the mixed experience. Firstly, the hotel's prime location stood out to me as a major advantage. Being close to many of Lisbon's main attractions and cities made exploring the area incredibly convenient. Additionally, the restaurant offered a picturesque view, and I particularly enjoyed the octopus salad we had for dinner. The abundance of breakfast choices was also a plus, even though I wished the quality of the food matched the variety. Finally, the staff were friendly and accommodating, which added a welcoming touch to our stay.",United States,Couple
1,2024-05-08,6.0,en-us,Disliked: Not value for money. For the money you pay you'd expect at least a free bottle of water in the room. Breakfast is below average. Hotel is old and is a bit outdated. Doesn't have enough facilities (ex: Gym),Location is very central. Beds are comfortable and shower was good. Staff were great and very helpful!,United Kingdom,Couple
2,2024-04-30,6.0,de,Disliked: A hotel that has seen better days and is in urgent need of renovation. The rooms are worn and you simply don't feel comfortable. Cleanliness also leaves much to be desired. Access to the beautiful rooftop terrace should always be guaranteed for hotel guests.,The hotel stands out for its location and the beautiful rooftop terrace.,Germany,Couple
3,2024-04-23,6.0,de,"Disliked: Stairs made of black stone. The edge is barely visible… potentially dangerous! It should be painted a light color. Large, many guests, breakfast has a high noise level, crowding 😰 The reception is often hectic as well.",Location,Switzerland,Solo traveller
4,2024-05-12,6.0,da,Disliked: It was too expensive for what you got; it should probably be a 3-star hotel if you only looked at the rooms.,The location,Denmark,Group
5,2024-04-29,6.0,en,Disliked: Wait time with bar staff sometimes considerably long in the lounge bar,Roof top bar and location,United Kingdom,Group
6,2024-05-19,7.0,nl,Disliked: The bathroom is dated. If you're a bit taller you cannot sit properly on the toilet. You shower in the bathtub (which is very slippery). If you're a bit taller you don't fit well under the showerhead. There were two holes in the floor (drains?). As a result there was a sewer smell in the bathroom. We solved this by placing the bath mat over the two covers. Given the price of the room this should not happen.,The staff are more than friendly and helpful. You really feel welcome as a guest. The breakfast is excellent. The upstairs restaurant is also worth it. The hotel is located close to public transport. No request or effort is too much. Every day a bottle of water was placed in the room.,Netherlands,Couple
7,2024-04-26,7.0,en-us,"Disliked: The fact that we tried numerous times to get a table at the Rooftop bar and got nowhere. In my opinion, preference should be given to guests that actually stay in the hotel and not to the public. One of the main reasons we chose this hotel was the Rooftop bar.

In [41]:
managerial_df = pd.DataFrame(
    agent_context[
        "managerial_evidence"
    ]
)

display(
    managerial_df[
        [
            "area",
            "positive_mentions",
            "negative_mentions",
            "net_mentions",
            "positive_share",
            "negative_share",
            "has_expectation_violation",
            "is_emerging_negative",
            "evidence_pattern",
        ]
    ]
)

,area,positive_mentions,negative_mentions,net_mentions,positive_share,negative_share,has_expectation_violation,is_emerging_negative,evidence_pattern
0,Breakfast,34,16,18,0.680,0.320,False,False,predominantly_positive
1,Restaurant and bar,20,13,7,0.606,0.394,True,True,positive_with_friction
2,Price–value,6,12,-6,0.333,0.667,True,True,net_negative
3,Room comfort,8,11,-3,0.421,0.579,False,False,net_negative
4,Bathroom condition,5,9,-4,0.357,0.643,True,False,net_negative
5,Rooftop access,24,8,16,0.750,0.250,True,True,positive_with_friction
6,Staff and service,46,8,38,0.852,0.148,False,False,predominantly_positive
7,Cleanliness and housekeeping,14,5,9,0.737,0.263,True,False,positive_with_friction


In [42]:
vulnerabilities_df = pd.DataFrame(
    agent_context[
        "diagnostic_vulnerabilities"
    ]
)

display(
    vulnerabilities_df[
        [
            "area",
            "evidence_pattern",
            "positive_mentions",
            "negative_mentions",
            "negative_share",
            "has_expectation_violation",
            "is_emerging_negative",
            "diagnostic_score",
            "diagnostic_score_components",
        ]
    ]
)

,area,evidence_pattern,positive_mentions,negative_mentions,negative_share,has_expectation_violation,is_emerging_negative,diagnostic_score,diagnostic_score_components
0,Price–value,net_negative,6,12,0.667,True,True,24,"{'negative_frequency': 10, 'net_negative': 3, 'negative_share': 2, 'expectation_violation': 4, 'emerging_negative': 3, 'default_priority': 2}"
1,Bathroom condition,net_negative,5,9,0.643,True,False,20,"{'negative_frequency': 9, 'net_negative': 3, 'negative_share': 2, 'expectation_violation': 4, 'emerging_negative': 0, 'default_priority': 2}"
2,Restaurant and bar,positive_with_friction,20,13,0.394,True,True,17,"{'negative_frequency': 10, 'net_negative': 0, 'negative_share': 0, 'expectation_violation': 4, 'emerging_negative': 3, 'default_priority': 0}"
3,Room comfort,net_negative,8,11,0.579,False,False,16,"{'negative_frequency': 10, 'net_negative': 3, 'negative_share': 1, 'expectation_violation': 0, 'emerging_negative': 0, 'default_priority': 2}"
4,Rooftop access,positive_with_friction,24,8,0.250,True,True,15,"{'negative_frequency': 8, 'net_negative': 0, 'negative_share': 0, 'expectation_violation': 4, 'emerging_negative': 3, 'default_priority': 0}"
5,Cleanliness and housekeeping,positive_with_friction,14,5,0.263,True,False,11,"{'negative_frequency': 5, 'net_negative': 0, 'negative_share': 0, 'expectation_violation': 4, 'emerging_negative': 0, 'default_priority': 2}"


In [43]:
strengths_df = pd.DataFrame(
    agent_context[
        "strengths"
    ]
)

display(
    strengths_df
)

,area,positive_mentions,negative_mentions,positive_share,strength_type,has_expectation_violation,is_emerging_negative
0,Staff and service,46,8,0.852,clear_strength,False,False
1,Breakfast,34,16,0.680,clear_strength,False,False
2,Rooftop access,24,8,0.750,strength_with_friction,True,True
3,Cleanliness and housekeeping,14,5,0.737,strength_with_friction,True,False


In [44]:
complaints_df = pd.DataFrame(
    agent_context[
        "complaint_evidence"
    ]
)

display(
    complaints_df[
        [
            "review_date",
            "rating",
            "original_language",
            "complaint",
            "positive_context",
        ]
    ]
)

,review_date,rating,original_language,complaint,positive_context
0,2024-04-26,5.0,en-us,"Disliked: Reflecting on my stay at Hotel Mundial, there were several aspects that left me feeling underwhelmed. Firstly, the room and bathroom fell short of my expectations, with outdated furnishings and a cramped layout. The shower posed a safety concern due to its slippery floor and constant leakage, making it difficult to use comfortably. Additionally, the bedding arrangement, with two twin beds pushed together instead of a king bed as requested, was disappointing. The lack of amenities such as a steamer or iron further detracted from the overall comfort of the room. Despite the hotel's convenient location, the outdated facilities and lackluster amenities made it difficult for me to overlook these shortcomings. Overall, while there were some positive aspects to my stay, I found the hotel to be lacking in terms of value and comfort.","In terms of what I liked about Hotel Mundial, there were definitely some positives to highlight amidst the mixed experience. Firstly, the hotel's prime location stood out to me as a major advantage. Being close to many of Lisbon's main attractions and cities made exploring the area incredibly convenient. Additionally, the restaurant offered a picturesque view, and I particularly enjoyed the octopus salad we had for dinner. The abundance of breakfast choices was also a plus, even though I wished the quality of the food matched the variety. Finally, the staff were friendly and accommodating, which added a welcoming touch to our stay."
1,2024-05-08,6.0,en-us,Disliked: Not value for money. For the money you pay you'd expect at least a free bottle of water in the room. Breakfast is below average. Hotel is old and is a bit outdated. Doesn't have enough facilities (ex: Gym),Location is very central. Beds are comfortable and shower was good. Staff were great and very helpful!
2,2024-04-30,6.0,de,Disliked: A hotel that has seen better days and is in urgent need of renovation. The rooms are worn and you simply don't feel comfortable. Cleanliness also leaves much to be desired. Access to the beautiful rooftop terrace should always be guaranteed for hotel guests.,The hotel stands out for its location and the beautiful rooftop terrace.
3,2024-04-23,6.0,de,"Disliked: Stairs made of black stone. The edge is barely visible… potentially dangerous! It should be painted a light color. Large, many guests, breakfast has a high noise level, crowding 😰 The reception is often hectic as well.",Location
4,2024-05-12,6.0,da,Disliked: It was too expensive for what you got; it should probably be a 3-star hotel if you only looked at the rooms.,The location
5,2024-04-29,6.0,en,Disliked: Wait time with bar staff sometimes considerably long in the lounge bar,Roof top bar and location
6,2024-05-19,7.0,nl,Disliked: The bathroom is dated. If you're a bit taller you cannot sit properly on the toilet. You shower in the bathtub (which is very slippery). If you're a bit taller you don't fit well under the showerhead. There were two holes in the floor (drains?). As a result there was a sewer smell in the bathroom. We solved this by placing the bath mat over the two covers. Given the price of the room this should not happen.,The staff are more than friendly and helpful. You really feel welcome as a guest. The breakfast is excellent. The upstairs restaurant is also worth it. The hotel is located close to public transport. No request or effort is too much. Every day a bottle of water was placed in the room.
7,2024-04-26,7.0,en-us,"Disliked: The fact that we tried numerous times to get a table at the Rooftop bar and got nowhere. In my opinion, preference should be given to guests that actually stay in the hotel and not to the public. One of the main reasons we chose this hotel was the Rooftop bar. Room service was non existent.","We loved the location. We loved the spacious sunny balcony. The complimentary bottle of wine and Portuguese pastry was a ni

# Decision_Agent

In [62]:
import importlib
import src.decision_agent

importlib.reload(
    src.decision_agent
)

from src.decision_agent import (
    run_decision_agent
)

In [63]:
decision = run_decision_agent(
    agent_context
)

In [64]:
decision[
    "overall_risk"
]

'moderate'

In [65]:
print(
    decision[
        "executive_summary"
    ]
)

Hotel Mundial shows clear strengths in staff/service and breakfast, and the rooftop remains a valued asset. However, four clusters require managerial attention: (1) bathroom safety and usability (urgent); (2) guest perception of price–value (review and alignment); (3) rooftop/restaurant access and reservations (preserve strength while resolving access friction); and (4) room condition and housekeeping (assess renovation need and housekeeping processes). A fifth, lower-priority item concerns bar service wait times and night-time noise/soundproofing. Recommendations emphasize inspection, policy/process review, and targeted operational fixes; where the evidence implies possible capital scope (room/bathroom renovation) we recommend assessment before committing investment.


In [66]:
priorities_df = pd.DataFrame(
    decision[
        "priorities"
    ]
)

display(
    priorities_df[
        [
            "rank",
            "area",
            "evidence_pattern",
            "managerial_priority",
            "intervention_type",
            "problem",
            "recommended_action",
            "safety_relevance",
            "requires_further_assessment",
            "confidence",
        ]
    ]
)

,rank,area,evidence_pattern,managerial_priority,intervention_type,problem,recommended_action,safety_relevance,requires_further_assessment,confidence
0,1,Bathroom condition (safety & usability),net_negative,urgent,mixed,"Multiple guests report slippery bathtub/shower surfaces, leaking/sloped showers, broken drains/holes, sewer smell and shower height issues; at least one report describes a potentially dangerous slippery floor.","Immediately inspect recently-reported rooms and public bathrooms for slipping hazards, leaks, blocked or damaged drains, and floor openings; implement urgent remediation where safety hazard is confirmed (temporary anti-slip mats/signage, repairs). Concurrently, audit bathroom amenities (toiletries, functioning drains, showerheads). If inspection suggests structural/renovation needs, commission a technical assessment to scope required capital work.",True,True,0.88
1,2,Price–value perception,net_negative,high,policy_process,"A notable share of guests perceive the stay as 'overpriced' given room condition and included amenities (water, parking). Expectation violations and emerging negative topic 'overpriced' are present.","Review pricing and value positioning relative to room condition and included guest amenities. Audit what is communicated and included (e.g., complimentary water, parking policy) and compare to guest expectations; consider updating promotional messaging, inclusions, or packaging if misalignment is confirmed. If pricing is to be maintained, strengthen value communication and on-property inclusions to reduce expectation gap.",False,True,0.76
2,3,Rooftop / Restaurant & bar access and reservations,positive_with_friction,high,policy_process,"Guests value the rooftop and rooftop restaurant/bar but report difficulties reserving tables, long wait times, and feeling that non-guests are prioritized over hotel guests, causing expectation violations despite overall positive sentiment for the rooftop.","Review rooftop access and reservation policies to ensure alignment with guest expectations and commercial objectives. Evaluate whether guest-priority access, reservation allocation, or clearer communication about rooftop availability and reservation procedures is appropriate. Improve front-desk and F&B coordination about reservations and guest guidance. Monitor wait times at peak periods and adjust operational staffing/queues as needed.",False,False,0.78
3,4,"Room condition, cleanliness and housekeeping",mixed,high,mixed,"Multiple guests describe dated rooms, worn furnishings, noisy rooms/poor soundproofing, and inconsistent housekeeping quality (carpet hairs, worn towels, inadequate cleaning). Some comments question whether current room standards match a 4-star positioning.","Short term: audit housekeeping processes, laundry/towel quality and room-cleaning checks to address immediate cleanliness lapses. Medium term: commission a condition assessment of room finishes and soundproofing to determine scope and cost of renovation or targeted refresh (prioritize bathrooms where safety issues were also reported). Use findings to align room category descriptions and pricing with actual condition or to plan refurbishment.",False,True,0.73
4,5,Bar service wait times & night-time noise / soundproofing,positive_with_friction,medium,operational,Guests report long wait times for bar service and problems with night-time noise due to poor soundproofing; these friction points reduce enjoyment of on-site F&B and room rest.,"Operationally review F&B staffing/peak-service procedures to reduce lounge/bar wait times and improve guest flow. Short-term actions could include queue management and clearer communication on expected wait. For noise complaints, perform a targeted assessment of soundproofing in frequently-complained rooms and, where feasible, deploy interim guest-facing mitigations (earplugs, room allocation guidance) while planning medium-term improvements.",False,False,0.62


In [67]:
for priority in decision[
    "priorities"
]:
    print("=" * 80)

    print(
        f"#{priority['rank']} "
        f"{priority['area']}"
    )

    print()

    print("EVIDENCE BASIS")
    print(
        json.dumps(
            priority[
                "evidence_basis"
            ],
            indent=2,
            ensure_ascii=False,
        )
    )

    print()

    print("SUPPORTING EVIDENCE")

    for evidence in priority[
        "supporting_evidence"
    ]:
        print(
            "-",
            evidence
        )

    print()

    print("POSITIVE COUNTEREVIDENCE")

    for evidence in priority[
        "positive_counterevidence"
    ]:
        print(
            "-",
            evidence
        )

    print()

#1 Bathroom condition (safety & usability)

EVIDENCE BASIS
{
  "positive_mentions": 5,
  "negative_mentions": 9,
  "expectation_violation": true,
  "emerging_negative": false,
  "experience_weeks": 0,
  "expectation_weeks": 1,
  "violated_expectation_weeks": 1,
  "emerging_negative_weeks": 0,
  "review_examples_used": 4,
  "relevant_trend_signals": [
    "emerging_negative_topic: shower experience",
    "emerging_negative_topic: toilet comfort",
    "matched diagnostic: Bathroom condition net_negative"
  ]
}

SUPPORTING EVIDENCE
- 2024-04-26: 'The shower posed a safety concern due to its slippery floor and constant leakage.'
- 2024-05-19: 'bathtub (which is very slippery)... there were two holes in the floor (drains?)... sewer smell in the bathroom.'
- 2024-05-07: 'The bathtub drain was not working.'
- Multiple managerial_evidence entries: Bathroom condition — 14 mentions (5 positive, 9 negative), evidence_pattern net_negative; has expectation violation = true.

POSITIVE COUNTEREVIDENC

In [68]:
monitoring_df = pd.DataFrame(
    decision[
        "monitoring_indicators"
    ]
)

display(
    monitoring_df[
        [
            "indicator",
            "reason",
            "desired_direction",
            "monitoring_horizon",
        ]
    ]
)

,indicator,reason,desired_direction,monitoring_horizon
0,EXPECTATION_EXPERIENCE_GAP_AVG,"Has increased over the lookback and signals widening gap between what is promised/expected and delivered; relevant to price–value, room condition and rooftop access.",decrease,short_term
1,SEMANTIC_MISALIGNMENT_SCORE_AVG,Rising misalignment suggests marketing/communication may not match actual guest experience (relevant to rooftop access and price/value messaging).,decrease,short_term
2,SENTIMENT_SCORE_AVG,Overall sentiment trended down over the window (with recent reversal); track for deterioration or improvement after interventions.,increase,ongoing
3,"Frequency of bathroom-related complaints (slippery, leaks, drains, sewer smell)",Directly tracks safety/usability priority—immediate remediation should reduce these mentions.,decrease,immediate
4,Mentions of 'overpriced' / price-value complaints,Tracks perception of value and effectiveness of pricing/communication changes.,decrease,short_term


In [69]:
strengths_decision_df = pd.DataFrame(
    decision[
        "strengths_to_preserve"
    ]
)

display(
    strengths_decision_df
)

,area,reason,management_implication
0,Staff and service,"High positive mentions (46 positive, positive_share 0.852). Multiple reviews praise friendly, helpful, and welcoming staff—this is a clear strength.",Maintain service standards and recognition/training programs; ensure front-desk and F&B staff have clear processes for reservations and guest prioritization to avoid undermining this strength.
1,Breakfast,"Strong positive signal (34 positive mentions, positive_share 0.68) and repeated guest praise for abundant choices.",Preserve breakfast variety and quality; consider minor adjustments if specific regional specialties are requested (monitor feedback).
2,Rooftop (asset),High positive sentiment for rooftop terrace and restaurant (24 positive mentions); perceived as a differentiator and reason guests book.,Protect rooftop guest experience by resolving access/reservation friction and ensuring F&B service consistency so the rooftop continues to deliver positive impact.


In [70]:
monitoring_df = pd.DataFrame(
    decision[
        "monitoring_indicators"
    ]
)

display(
    monitoring_df
)

,indicator,reason,desired_direction,monitoring_horizon
0,EXPECTATION_EXPERIENCE_GAP_AVG,"Has increased over the lookback and signals widening gap between what is promised/expected and delivered; relevant to price–value, room condition and rooftop access.",decrease,short_term
1,SEMANTIC_MISALIGNMENT_SCORE_AVG,Rising misalignment suggests marketing/communication may not match actual guest experience (relevant to rooftop access and price/value messaging).,decrease,short_term
2,SENTIMENT_SCORE_AVG,Overall sentiment trended down over the window (with recent reversal); track for deterioration or improvement after interventions.,increase,ongoing
3,"Frequency of bathroom-related complaints (slippery, leaks, drains, sewer smell)",Directly tracks safety/usability priority—immediate remediation should reduce these mentions.,decrease,immediate
4,Mentions of 'overpriced' / price-value complaints,Tracks perception of value and effectiveness of pricing/communication changes.,decrease,short_term


# A - BASELINE FREQUENCY

In [76]:
import importlib
import src.baseline_frequency

importlib.reload(
    src.baseline_frequency
)

from src.baseline_frequency import (
    run_frequency_baseline
)

In [77]:
frequency_decision = (
    run_frequency_baseline(
        agent_context
    )
)

In [78]:
frequency_df = pd.DataFrame(
    frequency_decision[
        "priorities"
    ]
)

display(
    frequency_df
)

,rank,area,managerial_domain,managerial_priority,negative_mentions,positive_mentions,negative_share,rationale
0,1,Breakfast,food_and_beverage,high,16,34,0.3200,Breakfast was ranked #1 because it contains 16 negative review mentions.
1,2,Restaurant and bar,food_and_beverage,medium,13,20,0.3939,Restaurant and bar was ranked #2 because it contains 13 negative review mentions.
2,3,Price–value,pricing_and_value,medium,12,6,0.6667,Price–value was ranked #3 because it contains 12 negative review mentions.
3,4,Room comfort,rooms_and_maintenance,low,11,8,0.5789,Room comfort was ranked #4 because it contains 11 negative review mentions.
4,5,Bathroom condition,rooms_and_maintenance,low,9,5,0.6429,Bathroom condition was ranked #5 because it contains 9 negative review mentions.


# B - GENERIC PIPELINE

In [81]:
import importlib
import src.baseline_generic_llm

importlib.reload(
    src.baseline_generic_llm
)

from src.baseline_generic_llm import (
    build_generic_context,
    run_generic_llm_baseline,
)

In [82]:
generic_context = (
    build_generic_context(
        case,
        max_reviews=12,
    )
)

print(
    json.dumps(
        generic_context,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "case": {
    "case_id": null,
    "decision_week": null,
    "decision_date": null,
    "window_start": null,
    "window_end": null
  },
  "hotel": {},
  "basic_performance": {
    "REVPAR_WEEK": {
      "first_value": 137.79088647884825,
      "last_value": 154.48296533177387,
      "percentage_change": 12.114065944040469
    },
    "REVPOR_WEEK": {
      "first_value": 211.5028830434784,
      "last_value": 261.13567753001746,
      "percentage_change": 23.466722425876398
    },
    "TREVPAR_WEEK": {
      "first_value": 183.41268953229107,
      "last_value": 209.317862853041,
      "percentage_change": 14.123980945271041
    },
    "TAXA_OCUPACAO": {
      "first_value": 89.11174785100286,
      "last_value": 87.39255014326648,
      "percentage_change": -1.929260450160766
    },
    "AVG_RATING": {
      "first_value": 8.808510638297872,
      "last_value": 8.575,
      "percentage_change": -2.650966183574878
    },
    "N_REVIEWS": {
      "first_value": 47.0,
      "last_v

In [83]:
generic_decision = (
    run_generic_llm_baseline(
        case,
        max_reviews=12,
    )
)

In [84]:
generic_df = pd.DataFrame(
    generic_decision[
        "priorities"
    ]
)

display(
    generic_df[
        [
            "rank",
            "area",
            "managerial_priority",
            "action_horizon",
            "intervention_type",
            "problem",
            "recommended_action",
            "safety_relevance",
            "requires_further_assessment",
            "confidence",
        ]
    ]
)

,rank,area,managerial_priority,action_horizon,intervention_type,problem,recommended_action,safety_relevance,requires_further_assessment,confidence
0,1,"Guest safety in wet/uneven areas (bathrooms, stairs)",urgent,immediate,maintenance,"Multiple reviewers reported slippery showers and bathtub surfaces, leaking showers, poorly visible black-stone stair edges and holes/drains in bathroom floors creating slip and trip hazards and sewer smell.","Conduct an immediate property-wide safety inspection of all bathrooms and public stairways. Implement temporary mitigations at once: anti-slip bath mats and adhesive non-slip strips in tubs/showers, clear signage where surfaces are slippery, temporary covers/guards for exposed holes/drains, install high-visibility nosing/strips on stair edges, and close or restrict any room or public area with severe hazards until repaired. Commission prioritized repairs for leaking showers, faulty drains and permanent anti-slip surfacing; schedule plumbing trades to resolve drainage/sewer issues.",True,True,0.90
1,2,Room and bathroom condition / renovation,high,medium_term,capital_assessment,"Numerous guests describe rooms and bathrooms as dated, cramped or worn (old furnishings, worn towels, poor bathroom layout and low shower head), which affects comfort and star-standard expectations.","Commission a condition survey and costed phased refurbishment plan prioritizing bathrooms (drains, showerheads, anti-slip surfacing, plumbing), mattresses and bedding configuration (ability to provide requested king beds rather than pushed twins), worn soft goods replacement (towels, carpets where needed), and room layout improvements where feasible. Use a phased rollout to avoid taking large room blocks offline at once; prioritize rooms with the worst reviews and those used for higher rate segments.",True,True,0.85
2,3,Housekeeping and cleanliness standards,high,short_term,operational,"Several reviews note inadequate room cleaning (carpet hairs, worn towels, inconsistent bathroom cleanliness and non-working drains), leading to perceptions of poor hygiene and lower guest satisfaction.","Perform an immediate housekeeping standards audit (spot checks across room types and floors). Retrain staff on cleaning checklists, focusing on carpets, towel inspection/replacement, bath/tub drains and odor control. Institute shift handover inspections, daily room audits with scoring, and a rapid response protocol for guest cleanliness complaints. Replace worn towels and damaged linens as a short-term investment. Track corrective action closure times.",False,True,0.85
3,4,Rooftop bar/terrace access and food & beverage service capacity,medium,short_term,policy_process,"Guests report difficulty getting rooftop bar tables (public given priority over hotel guests), long wait times at the lounge bar, and inconsistent room service — causing frustration because the rooftop is a key reason guests choose the hotel.",Implement a guest-priority policy for the rooftop (reserve a percentage of seating or a guest-only booking window). Introduce a simple reservation path for hotel guests at check-in and via the hotel app/concierge. Assess peak staffing levels at the rooftop/bar and adjust staffing or service processes to reduce wait times. Clarify and publicize room service availability and service standards at check-in.,False,True,0.70
4,5,"Value perception, amenities and pricing alignment",medium,medium_term,policy_process,"Multiple guests feel the hotel does not deliver value for the price paid (requests for basic inclusions like bottled water, complaints that facilities are limited, no gym), and some rate the breakfast below expectations despite others praising it.","Conduct a simple competitive benchmarking exercise to confirm star positioning and price parity. Consider low-cost amenity changes that improve perceived value: complimentary bottled water in rooms (where inconsistent), offering basic in-room iron/steamer on request or a loaner program

In [85]:
print(
    generic_decision[
        "executive_summary"
    ]
)

Recent guest reviews consistently praise the hotel's location, rooftop views and friendly staff but highlight recurring problems: bathroom safety and drainage issues, outdated rooms needing renovation, inconsistent cleanliness/housekeeping, crowding and access problems at the rooftop/bar and perception of poor value for price. Immediate actions should address documented safety hazards and housekeeping; medium-term work should plan phased bathroom/room refurbishment and align pricing/amenities. Preserve strong location, rooftop asset, restaurant and staff responsiveness while monitoring guest ratings, safety incidents, housekeeping audit scores and rooftop guest access metrics.


In [86]:
pd.DataFrame(
    generic_decision[
        "strengths_to_preserve"
    ]
)

,area,reason,management_implication
0,Location,"Multiple reviews highlight the hotel's prime, central location and convenience for transport and attractions.","Leverage location in marketing, ensure concierge and front desk maintain strong local knowledge and transport guidance; prioritize refurb works to preserve guest access and minimize disruption."
1,Rooftop terrace and restaurant views,Guests repeatedly cite the rooftop terrace and upstairs restaurant (views and food) as standout positives.,"Protect rooftop as a guest-facing asset (see priority 4), maintain F&B quality and atmosphere while improving access control for guests."
2,Staff friendliness and service attitude,"Many guests praise friendly, helpful staff and front office efficiency.",Continue staff training and recognition programs; leverage staff strength during periods of change to maintain guest goodwill.
3,Breakfast offering (abundance/variety),"Several reviews praise abundant and excellent breakfast, even where some guests found it below average.",Maintain core breakfast variety while addressing quality and adding regional items selectively.
